# Manufacturing Efficiency Intelligence
## Data Profiling, Cleaning & Validation

### Objective
This notebook prepares the manufacturing dataset for analysis by profiling, cleaning, and validating the raw data while preserving valid operational behavior.

### Data Preparation Approach
The process will follow these stages:

1. Raw Data Loading
2. Structural Profiling
3. Data Type Standardization
4. Missing Value Assessment
5. Duplicate Validation
6. Business Rule Validation
7. Time-Series Integrity Checks
8. Referential Integrity Validation
9. Outlier Assessment
10. Data Quality Flagging
11. Normalization
12. Feature Engineering
13. Final Data Quality Validation
14. Export to SQL Server

> **Important:** Raw data will remain unchanged. Cleaning decisions will distinguish between actual data-quality issues, valid business nulls, and meaningful operational anomalies.

# 1. Environment Setup & Raw Data Loading

## Objective
Set up the Python environment, define project paths, and load the raw manufacturing datasets without modifying the original source files.

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

In [10]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

### 1.1 Define Project Paths

The project uses separate directories for raw data, processed data, and generated outputs.  
Raw source files are treated as read-only and will not be modified during the preparation process.

### 1.2 Define Project Paths

Separate raw data, processed data, and generated outputs to preserve the original source files and maintain a reproducible workflow.

In [11]:
RAW_PATH = Path(r"D:\Manufacturing_Efficiency_Project\Data\Row")
PROCESSED_PATH = Path(r"D:\Manufacturing_Efficiency_Project\Data\Processed")
OUTPUT_PATH = Path(r"D:\Manufacturing_Efficiency_Project\Outputs")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

In [12]:
print("Raw path exists:", RAW_PATH.exists())
print("Processed path exists:", PROCESSED_PATH.exists())
print("Output path exists:", OUTPUT_PATH.exists())

Raw path exists: True
Processed path exists: True
Output path exists: True


### 1.3 Verify Raw Source Files

Verify that the expected raw CSV files are available before loading them into memory.  
This prevents missing-file or incorrect-path issues from affecting the subsequent workflow.

In [13]:
expected_files = [
    "Fact_ProductionEnergy.csv",
    "Fact_DowntimeEvents.csv",
    "Dim_Plant.csv",
    "Dim_Shift.csv",
    "Dim_Status.csv",
    "Dim_EnergyTariff.csv",
    "Dim_DowntimeReason.csv"
]

for file_name in expected_files:
    file_path = RAW_PATH / file_name
    print(f"{file_name}: {'Found' if file_path.exists() else 'Missing'}")

Fact_ProductionEnergy.csv: Found
Fact_DowntimeEvents.csv: Found
Dim_Plant.csv: Found
Dim_Shift.csv: Found
Dim_Status.csv: Found
Dim_EnergyTariff.csv: Found
Dim_DowntimeReason.csv: Found


### 1.4 Load Dimension Tables

Load the reference tables containing descriptive information about plants, shifts, operating statuses, energy tariffs, and downtime reasons.

These tables are relatively small and will later support business-rule and referential-integrity validation.

In [14]:
dim_plant = pd.read_csv(RAW_PATH / "Dim_Plant.csv")
dim_shift = pd.read_csv(RAW_PATH / "Dim_Shift.csv")
dim_status = pd.read_csv(RAW_PATH / "Dim_Status.csv")
dim_tariff = pd.read_csv(RAW_PATH / "Dim_EnergyTariff.csv")
dim_downtime_reason = pd.read_csv(RAW_PATH / "Dim_DowntimeReason.csv")

### 1.5 Load Fact Tables

Load the main production-energy and downtime-event fact tables.

The production-energy table is the largest dataset in the project and represents the primary source of detailed operational observations.

In [15]:
fact_production = pd.read_csv(
    RAW_PATH / "Fact_ProductionEnergy.csv",
    low_memory=False
)

fact_downtime = pd.read_csv(
    RAW_PATH / "Fact_DowntimeEvents.csv",
    low_memory=False
)

### 1.6 Create Dataset Registry

Organize the loaded DataFrames in a single registry to simplify repeated profiling and validation operations across all datasets.

In [16]:
datasets = {
    "Fact_ProductionEnergy": fact_production,
    "Fact_DowntimeEvents": fact_downtime,
    "Dim_Plant": dim_plant,
    "Dim_Shift": dim_shift,
    "Dim_Status": dim_status,
    "Dim_EnergyTariff": dim_tariff,
    "Dim_DowntimeReason": dim_downtime_reason
}

print(f"Successfully loaded {len(datasets)} datasets.")

Successfully loaded 7 datasets.


# 2. Initial Structural Profiling

## Objective
Assess the structure, size, memory footprint, schema, and basic content of the loaded datasets before applying any cleaning or transformation.

This stage establishes an initial understanding of the data and helps identify potential structural issues that require further investigation.

> **Note:** No data values are modified during this stage.

### 2.1 Dataset Dimensions

Review the number of rows and columns in each dataset to understand the overall scale and structure of the available data.

In [17]:
shape_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

shape_summary

,Dataset,Rows,Columns
0,Fact_ProductionEnergy,2947392,23
1,Fact_DowntimeEvents,13046,11
2,Dim_Plant,2,6
3,Dim_Shift,3,4
4,Dim_Status,4,4
5,Dim_EnergyTariff,8,8
6,Dim_DowntimeReason,6,5


### 2.2 Memory Usage

Measure the memory footprint of each dataset to identify large tables and support efficient processing decisions during cleaning and transformation.

In [18]:
memory_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Memory_MB": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2),
            2
        )
    }
    for name, df in datasets.items()
])

memory_summary.sort_values(
    "Memory_MB",
    ascending=False
).reset_index(drop=True)

,Dataset,Memory_MB
0,Fact_ProductionEnergy,882.15
1,Fact_DowntimeEvents,2.05
2,Dim_Plant,0.00
3,Dim_Shift,0.00
4,Dim_Status,0.00
5,Dim_EnergyTariff,0.00
6,Dim_DowntimeReason,0.00


### 2.3 Preview Fact Tables

Inspect sample records from the main fact tables to understand their structure, field values, and general record format before detailed profiling.

In [19]:
display(fact_production.head())
display(fact_downtime.head())

,ReadingID,Timestamp,PlantID,ShiftID,TariffID,StatusID,MachineID,MachineName,MachineType,LineID,LineName,ProductID,ProductName,ProductFamily,StandardCycleTimeSec,ItemsProduced,RejectedItems,ActualCycleTimeSec,PowerAvgKW,PowerMaxKW,PowerMinKW,EnergyKWh,AmbientTempC
0,1,2024-01-01 00:00:00,PLT-01,S3,T24-OFF,2,MCH-001,CNC Mill 01,CNC Milling,LN-01,Precision Machining A,PRD-002,Valve Body B,Valve Components,48.00,6,0,48.78,34.10,38.41,31.00,2.84,25.22
1,2,2024-01-01 00:05:00,PLT-01,S3,T24-OFF,2,MCH-001,CNC Mill 01,CNC Milling,LN-01,Precision Machining A,PRD-002,Valve Body B,Valve Components,48.00,5,1,50.93,33.46,38.06,30.07,2.79,24.73
2,3,2024-01-01 00:10:00,PLT-01,S3,T24-OFF,2,MCH-001,CNC Mill 01,CNC Milling,LN-01,Precision Machining A,PRD-002,Valve Body B,Valve Components,48.00,6,0,46.44,33.78,38.10,28.86,2.82,25.31
3,4,2024-01-01 00:15:00,PLT-01,S3,T24-OFF,2,MCH-001,CNC Mill 01,CNC Milling,LN-01,Precision Machining A,PRD-002,Valve Body B,Valve Components,48.00,5,0,49.99,33.62,38.18,28.86,2.80,25.31
4,5,2024-01-01 00:20:00,PLT-01,S3,T24-OFF,2,MCH-001,CNC Mill 01,CNC Milling,LN-01,Precision Machining A,PRD-002,Valve Body B,Valve Components,48.00,6,0,46.58,33.47,35.52,28.67,2.79,24.66


,DowntimeEventID,PlantID,MachineID,LineID,ShiftID,StartTimestamp,EndTimestamp,DurationMinutes,ReasonID,ProductID,EnergyDuringDowntimeKWh
0,DT-000001,PLT-01,MCH-001,LN-01,S3,2024-01-01 02:50:00,2024-01-01 03:05:00,15,R01,PRD-002,1.57
1,DT-000002,PLT-01,MCH-001,LN-01,S1,2024-01-01 06:35:00,2024-01-01 06:50:00,15,R06,PRD-001,1.79
2,DT-000003,PLT-01,MCH-001,LN-01,S3,2024-01-02 04:55:00,2024-01-02 05:05:00,10,R02,PRD-003,1.11
3,DT-000004,PLT-01,MCH-001,LN-01,S1,2024-01-02 11:25:00,2024-01-02 11:35:00,10,R01,PRD-010,1.22
4,DT-000005,PLT-01,MCH-001,LN-01,S3,2024-01-02 23:15:00,2024-01-02 23:20:00,5,R06,PRD-002,0.55


In [20]:
display(fact_production.tail())
display(fact_downtime.tail())


,ReadingID,Timestamp,PlantID,ShiftID,TariffID,StatusID,MachineID,MachineName,MachineType,LineID,LineName,ProductID,ProductName,ProductFamily,StandardCycleTimeSec,ItemsProduced,RejectedItems,ActualCycleTimeSec,PowerAvgKW,PowerMaxKW,PowerMinKW,EnergyKWh,AmbientTempC
2947387,2947388,2025-12-31 23:35:00,PLT-02,S3,T25-OFF2,2,MCH-014,Finishing Cell 03,Finishing Cell,LN-05,Finishing & Final Assembly,PRD-006,Rotor Sleeve,Shaft Components,51.00,5,0,50.28,20.51,22.17,18.09,1.71,24.55
2947388,2947389,2025-12-31 23:40:00,PLT-02,S3,T25-OFF2,2,MCH-014,Finishing Cell 03,Finishing Cell,LN-05,Finishing & Final Assembly,PRD-006,Rotor Sleeve,Shaft Components,51.00,5,0,52.84,22.07,24.79,18.91,1.84,24.23
2947389,2947390,2025-12-31 23:45:00,PLT-02,S3,T25-OFF2,2,MCH-014,Finishing Cell 03,Finishing Cell,LN-05,Finishing & Final Assembly,PRD-006,Rotor Sleeve,Shaft Components,51.00,5,0,50.04,20.67,22.12,18.57,1.72,24.57
2947390,2947391,2025-12-31 23:50:00,PLT-02,S3,T25-OFF2,2,MCH-014,Finishing Cell 03,Finishing Cell,LN-05,Finishing & Final Assembly,PRD-006,Rotor Sleeve,Shaft Components,51.00,5,0,54.50,21.50,23.31,18.36,1.79,25.74
2947391,2947392,2025-12-31 23:55:00,PLT-02,S3,T25-OFF2,2,MCH-014,Finishing Cell 03,Finishing Cell,LN-05,Finishing & Final Assembly,PRD-006,Rotor Sleeve,Shaft Components,51.00,5,0,54.09,21.01,23.01,18.16,1.75,25.17


,DowntimeEventID,PlantID,MachineID,LineID,ShiftID,StartTimestamp,EndTimestamp,DurationMinutes,ReasonID,ProductID,EnergyDuringDowntimeKWh
13041,DT-013042,PLT-02,MCH-014,LN-05,S3,2025-12-29 23:20:00,2025-12-29 23:25:00,5,R05,PRD-012,0.38
13042,DT-013043,PLT-02,MCH-014,LN-05,S3,2025-12-30 00:40:00,2025-12-30 00:45:00,5,R06,PRD-012,0.37
13043,DT-013044,PLT-02,MCH-014,LN-05,S1,2025-12-30 10:25:00,2025-12-30 10:45:00,20,R05,PRD-006,1.52
13044,DT-013045,PLT-02,MCH-014,LN-05,S3,2025-12-30 22:40:00,2025-12-30 22:45:00,5,R04,PRD-009,0.34
13045,DT-013046,PLT-02,MCH-014,LN-05,S3,2025-12-31 22:40:00,2025-12-31 23:10:00,30,R01,PRD-006,2.31


### 2.4 Column Names and Current Data Types

Review the schema of each dataset, including column names and the data types initially inferred by Pandas.

These inferred types will later be compared against the expected business data types before standardization.

In [21]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))

    display(
        pd.DataFrame({
            "Column": df.columns,
            "Current_DataType": df.dtypes.astype(str).values
        })
    )


Fact_ProductionEnergy
---------------------


,Column,Current_DataType
0,ReadingID,int64
1,Timestamp,str
2,PlantID,str
3,ShiftID,str
4,TariffID,str
5,StatusID,int64
6,MachineID,str
7,MachineName,str
8,MachineType,str
9,LineID,str



Fact_DowntimeEvents
-------------------


,Column,Current_DataType
0,DowntimeEventID,str
1,PlantID,str
2,MachineID,str
3,LineID,str
4,ShiftID,str
5,StartTimestamp,str
6,EndTimestamp,str
7,DurationMinutes,int64
8,ReasonID,str
9,ProductID,str



Dim_Plant
---------


...

,Column,Current_DataType
0,StatusID,int64
1,StatusName,str
2,IsProductive,int64
3,SeverityLevel,str



Dim_EnergyTariff
----------------


,Column,Current_DataType
0,TariffID,str
1,TariffName,str
2,CalendarYear,int64
3,TimeBand,str
4,StartTime,str
5,EndTime,str
6,RatePerKWh,float64
7,IsPeak,int64



Dim_DowntimeReason
------------------


,Column,Current_DataType
0,ReasonID,str
1,ReasonCategory,str
2,ReasonName,str
3,IsPlanned,int64
4,SeverityWeight,int64


### 2.5 Detailed Structural Information

Inspect non-null counts, current data types, and memory usage for the main fact tables to establish a detailed structural baseline.

In [22]:
fact_production.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 2947392 entries, 0 to 2947391
Data columns (total 23 columns):
 #   Column                Dtype  
---  ------                -----  
 0   ReadingID             int64  
 1   Timestamp             str    
 2   PlantID               str    
 3   ShiftID               str    
 4   TariffID              str    
 5   StatusID              int64  
 6   MachineID             str    
 7   MachineName           str    
 8   MachineType           str    
 9   LineID                str    
 10  LineName              str    
 11  ProductID             str    
 12  ProductName           str    
 13  ProductFamily         str    
 14  StandardCycleTimeSec  float64
 15  ItemsProduced         int64  
 16  RejectedItems         int64  
 17  ActualCycleTimeSec    float64
 18  PowerAvgKW            float64
 19  PowerMaxKW            float64
 20  PowerMinKW            float64
 21  EnergyKWh             float64
 22  AmbientTempC          float64
dtypes: float64(7), int

### 2.6 Review Dimension Tables

Inspect the complete contents of the small reference tables to understand the business entities, categories, and rules represented in the data.

This review will support subsequent business-rule and referential-integrity validation.

In [23]:
display(dim_plant)
display(dim_shift)
display(dim_status)
display(dim_tariff)
display(dim_downtime_reason)

,PlantID,PlantName,City,Country,GridRegion,IndustryType
0,PLT-01,First Plant,City 1,Egypt,Region 1,Discrete Manufacturing
1,PLT-02,Second Plant,City 2,Egypt,Region 2,Discrete Manufacturing


,ShiftID,ShiftName,StartTime,EndTime
0,S1,Morning,06:00,14:00
1,S2,Evening,14:00,22:00
2,S3,Night,22:00,06:00


,StatusID,StatusName,IsProductive,SeverityLevel
0,0,Idle,0,Low
1,1,Manual Production,1,Low
2,2,Automatic Production,1,Low
3,3,Alarm / Interrupted,0,High


,TariffID,TariffName,CalendarYear,TimeBand,StartTime,EndTime,RatePerKWh,IsPeak
0,T24-OFF,2024 Off-Peak,2024,Off-Peak,00:00,06:00,1.10,0
1,T24-SHO,2024 Shoulder,2024,Shoulder,06:00,17:00,1.40,0
2,T24-PEAK,2024 Peak,2024,Peak,17:00,22:00,1.80,1
3,T24-OFF2,2024 Late Off-Peak,2024,Off-Peak,22:00,24:00,1.10,0
4,T25-OFF,2025 Off-Peak,2025,Off-Peak,00:00,06:00,1.25,0
5,T25-SHO,2025 Shoulder,2025,Shoulder,06:00,17:00,1.55,0
6,T25-PEAK,2025 Peak,2025,Peak,17:00,22:00,2.05,1
7,T25-OFF2,2025 Late Off-Peak,2025,Off-Peak,22:00,24:00,1.25,0


,ReasonID,ReasonCategory,ReasonName,IsPlanned,SeverityWeight
0,R01,Maintenance,Planned Maintenance,1,2
1,R02,Mechanical,Mechanical Failure,0,5
2,R03,Electrical,Electrical Fault,0,5
3,R04,Tooling,Tooling Adjustment,1,2
4,R05,Material,Material Shortage,0,3
5,R06,Controls,Sensor / Control Fault,0,4


# 3. Data Quality Profiling

## Objective
Establish a baseline data-quality profile before applying any cleaning or transformation.

This stage evaluates:
- Missing values
- Exact duplicate records
- Primary-key completeness and uniqueness
- Column cardinality
- Numeric value ranges
- Categorical distributions
- Temporal coverage

> **Note:** Findings identified during this stage are not automatically treated as errors. They will be interpreted using business context before any cleaning decision is made.

### 3.1 Missing Value Profile

Quantify missing values across all datasets to identify columns that require further investigation.

At this stage, missing values are measured only. Their business meaning and appropriate treatment will be assessed separately.

In [24]:
def missing_profile(df):
    profile = pd.DataFrame({
        "Missing_Count": df.isna().sum(),
        "Missing_Percent": (df.isna().mean() * 100).round(2)
    })

    return profile[profile["Missing_Count"] > 0].sort_values(
        "Missing_Percent",
        ascending=False
    )

In [25]:
missing_profile(fact_production)

,Missing_Count,Missing_Percent
ActualCycleTimeSec,230462,7.82
ProductID,184906,6.27
ProductName,184906,6.27
ProductFamily,184906,6.27
StandardCycleTimeSec,184906,6.27


In [26]:
missing_profile(fact_downtime)

,Missing_Count,Missing_Percent


In [27]:
for name, df in datasets.items():
    missing = missing_profile(df)

    if not missing.empty:
        print(f"\n{name}")
        display(missing)


Fact_ProductionEnergy


,Missing_Count,Missing_Percent
ActualCycleTimeSec,230462,7.82
ProductID,184906,6.27
ProductName,184906,6.27
ProductFamily,184906,6.27
StandardCycleTimeSec,184906,6.27


### 3.2 Exact Duplicate Assessment

Check each dataset for fully duplicated records.

An exact duplicate occurs when all column values are identical across two or more rows. Duplicate records are identified at this stage but will not be removed until their validity is assessed.

In [28]:
duplicate_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Exact_Duplicates": df.duplicated().sum()
    }
    for name, df in datasets.items()
])

duplicate_summary

,Dataset,Exact_Duplicates
0,Fact_ProductionEnergy,0
1,Fact_DowntimeEvents,0
2,Dim_Plant,0
3,Dim_Shift,0
4,Dim_Status,0
5,Dim_EnergyTariff,0
6,Dim_DowntimeReason,0


### 3.3 Primary-Key Validation

Validate the completeness and uniqueness of the expected primary key in each dataset.

A valid primary key should:
1. Contain no missing values.
2. Uniquely identify every record.

In [29]:
primary_keys = {
    "Fact_ProductionEnergy": "ReadingID",
    "Fact_DowntimeEvents": "DowntimeEventID",
    "Dim_Plant": "PlantID",
    "Dim_Shift": "ShiftID",
    "Dim_Status": "StatusID",
    "Dim_EnergyTariff": "TariffID",
    "Dim_DowntimeReason": "ReasonID"
}

In [30]:
key_results = []

for name, key in primary_keys.items():
    df = datasets[name]

    key_results.append({
        "Dataset": name,
        "Primary_Key": key,
        "Missing_Keys": df[key].isna().sum(),
        "Duplicate_Keys": df[key].duplicated().sum(),
        "Unique_Values": df[key].nunique()
    })

key_validation = pd.DataFrame(key_results)

key_validation

,Dataset,Primary_Key,Missing_Keys,Duplicate_Keys,Unique_Values
0,Fact_ProductionEnergy,ReadingID,0,0,2947392
1,Fact_DowntimeEvents,DowntimeEventID,0,0,13046
2,Dim_Plant,PlantID,0,0,2
3,Dim_Shift,ShiftID,0,0,3
4,Dim_Status,StatusID,0,0,4
5,Dim_EnergyTariff,TariffID,0,0,8
6,Dim_DowntimeReason,ReasonID,0,0,6


### 3.4 Column Cardinality

Measure the number of distinct values in each column to better understand identifiers, categorical variables, low-cardinality attributes, and potentially inconsistent categories. 

In [31]:
cardinality_profile = pd.DataFrame({
    "Column": fact_production.columns,
    "Unique_Values": [
        fact_production[col].nunique(dropna=True)
        for col in fact_production.columns
    ]
})

cardinality_profile

,Column,Unique_Values
0,ReadingID,2947392
1,Timestamp,210528
2,PlantID,2
3,ShiftID,3
4,TariffID,8
5,StatusID,4
6,MachineID,14
7,MachineName,14
8,MachineType,4
9,LineID,5


### 3.5 Numeric Range Profiling

Review the statistical ranges of key numeric operational variables to identify values that may require subsequent business-rule or outlier validation.

Extreme values are not automatically classified as errors because they may represent meaningful operational behavior.

In [32]:
numeric_columns = [
    "StandardCycleTimeSec",
    "ItemsProduced",
    "RejectedItems",
    "ActualCycleTimeSec",
    "PowerAvgKW",
    "PowerMaxKW",
    "PowerMinKW",
    "EnergyKWh",
    "AmbientTempC"
]

fact_production[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
StandardCycleTimeSec,"2,762,486.00",48.76,6.82,38.00,42.00,48.00,53.00,62.00
ItemsProduced,"2,947,392.00",4.95,1.70,0.00,5.00,5.00,6.00,8.00
RejectedItems,"2,947,392.00",0.04,0.20,0.00,0.00,0.00,0.00,1.00
ActualCycleTimeSec,"2,716,930.00",50.07,7.44,34.65,44.03,49.39,55.23,81.79
PowerAvgKW,"2,947,392.00",27.39,9.79,1.80,20.66,30.46,35.31,42.73
PowerMaxKW,"2,947,392.00",30.13,10.79,1.91,22.71,33.49,38.76,48.68
PowerMinKW,"2,947,392.00",24.10,8.64,1.51,18.15,26.77,30.99,39.30
EnergyKWh,"2,947,392.00",2.28,0.82,0.15,1.72,2.54,2.94,3.56
AmbientTempC,"2,947,392.00",25.00,4.95,13.24,20.79,24.99,29.21,36.75


### 3.6 Categorical Distribution

Review the frequency distribution of key categorical variables to understand the operational composition of the production dataset and identify potentially inconsistent categories. 

In [33]:
categorical_columns = [
    "PlantID",
    "ShiftID",
    "TariffID",
    "StatusID",
    "MachineID",
    "MachineType",
    "LineID",
    "ProductID",
    "ProductFamily"
]

for col in categorical_columns:
    print(f"\n{col}")
    display(
        fact_production[col]
        .value_counts(dropna=False)
        .to_frame("Count")
    )


PlantID


,Count
PlantID,
PLT-01,1684224
PLT-02,1263168



ShiftID


,Count
ShiftID,
S3,982464
S1,982464
S2,982464



TariffID


...

,Count
LineID,
LN-01,631584
LN-02,631584
LN-04,631584
LN-05,631584
LN-03,421056



ProductID


,Count
ProductID,
PRD-012,309965
PRD-010,295933
PRD-001,267750
PRD-005,253732
PRD-007,253616
PRD-003,239692
PRD-011,239685
PRD-008,197432
PRD-009,197329



ProductFamily


,Count
ProductFamily,
Housing Components,845583
Valve Components,690692
Structural Components,648377
Shaft Components,577834
NaN,184906


### 3.7 Temporal Coverage

Assess the available time span and timestamp validity of the production and downtime datasets before applying formal datetime conversion and detailed time-series integrity checks.

In [34]:
production_timestamp_check = pd.to_datetime(
    fact_production["Timestamp"],
    errors="coerce"
)

print("Production Start:", production_timestamp_check.min())
print("Production End:", production_timestamp_check.max())
print("Invalid Timestamps:", production_timestamp_check.isna().sum())

Production Start: 2024-01-01 00:00:00
Production End: 2025-12-31 23:55:00
Invalid Timestamps: 0


In [35]:
downtime_start_check = pd.to_datetime(
    fact_downtime["StartTimestamp"],
    errors="coerce"
)

downtime_end_check = pd.to_datetime(
    fact_downtime["EndTimestamp"],
    errors="coerce"
)

print("Downtime Start:", downtime_start_check.min())
print("Downtime End:", downtime_end_check.max())

print(
    "Invalid Start Timestamps:",
    downtime_start_check.isna().sum()
)

print(
    "Invalid End Timestamps:",
    downtime_end_check.isna().sum()
)

Downtime Start: 2024-01-01 00:35:00
Downtime End: 2025-12-31 23:20:00
Invalid Start Timestamps: 0
Invalid End Timestamps: 0


# 4. Data Type Standardization

## Objective
Standardize column data types to ensure that each field is represented according to its business meaning and is suitable for reliable validation and analysis.

This stage focuses on:
- Datetime fields
- Integer fields
- Decimal numeric fields
- Time fields
- Identifier and categorical fields

> **Note:** Data-type conversion changes how values are represented and processed, but does not intentionally alter their business meaning.

### 4.1 Create Working Copies

Create separate working copies of the raw DataFrames before applying transformations.

This preserves the originally loaded data in memory and ensures that all cleaning and validation operations are performed on dedicated working datasets.

In [36]:
fact_production_clean = fact_production.copy()
fact_downtime_clean = fact_downtime.copy()

dim_plant_clean = dim_plant.copy()
dim_shift_clean = dim_shift.copy()
dim_status_clean = dim_status.copy()
dim_tariff_clean = dim_tariff.copy()
dim_downtime_reason_clean = dim_downtime_reason.copy()

### 4.2 Standardize Datetime Fields

Convert timestamp fields from text-based representations to Pandas datetime values.

Invalid datetime values are converted to missing datetime values (`NaT`) so they can be identified and handled explicitly during validation.

In [37]:
fact_production_clean["Timestamp"] = pd.to_datetime(
    fact_production_clean["Timestamp"],
    errors="coerce"
)

fact_downtime_clean["StartTimestamp"] = pd.to_datetime(
    fact_downtime_clean["StartTimestamp"],
    errors="coerce"
)

fact_downtime_clean["EndTimestamp"] = pd.to_datetime(
    fact_downtime_clean["EndTimestamp"],
    errors="coerce"
)

In [38]:
print(
    "Invalid Production Timestamps:",
    fact_production_clean["Timestamp"].isna().sum()
)

print(
    "Invalid Downtime Start Timestamps:",
    fact_downtime_clean["StartTimestamp"].isna().sum()
)

print(
    "Invalid Downtime End Timestamps:",
    fact_downtime_clean["EndTimestamp"].isna().sum()
)

Invalid Production Timestamps: 0
Invalid Downtime Start Timestamps: 0
Invalid Downtime End Timestamps: 0


### 4.3 Standardize Integer Fields

Convert count-based and identifier-related numeric fields to appropriate integer representations while preserving valid missing values where applicable.

In [39]:
production_integer_columns = [
    "StatusID",
    "StandardCycleTimeSec",
    "ItemsProduced",
    "RejectedItems"
]

for col in production_integer_columns:
    fact_production_clean[col] = pd.to_numeric(
        fact_production_clean[col],
        errors="coerce"
    ).astype("Int64")

In [40]:
fact_downtime_clean["DurationMinutes"] = pd.to_numeric(
    fact_downtime_clean["DurationMinutes"],
    errors="coerce"
).astype("Int64")

### 4.4 Standardize Decimal Numeric Fields

Convert continuous operational measurements to numeric decimal representations to support mathematical validation and analytical calculations.

In [41]:
production_decimal_columns = [
    "ActualCycleTimeSec",
    "PowerAvgKW",
    "PowerMaxKW",
    "PowerMinKW",
    "EnergyKWh",
    "AmbientTempC"
]

for col in production_decimal_columns:
    fact_production_clean[col] = pd.to_numeric(
        fact_production_clean[col],
        errors="coerce"
    )

In [42]:
fact_downtime_clean["EnergyDuringDowntimeKWh"] = pd.to_numeric(
    fact_downtime_clean["EnergyDuringDowntimeKWh"],
    errors="coerce"
)

### 4.5 Validate Energy Tariff Time Fields

Review tariff start and end times before time conversion.

The value `24:00` represents the end of the current day and requires special handling because standard datetime parsers may not accept it as a conventional time value.

In [43]:
display(
    dim_tariff_clean[
        ["TariffID", "StartTime", "EndTime"]
    ]
)

,TariffID,StartTime,EndTime
0,T24-OFF,00:00,06:00
1,T24-SHO,06:00,17:00
2,T24-PEAK,17:00,22:00
3,T24-OFF2,22:00,24:00
4,T25-OFF,00:00,06:00
5,T25-SHO,06:00,17:00
6,T25-PEAK,17:00,22:00
7,T25-OFF2,22:00,24:00


In [44]:
print(
    "24:00 EndTime records:",
    (dim_tariff_clean["EndTime"] == "24:00").sum()
)

24:00 EndTime records: 2


### 4.6 Standardize Boolean Indicators

Standardize binary indicator fields so that their values are represented consistently for subsequent validation and analysis.

In [45]:
print(
    "IsProductive:",
    dim_status_clean["IsProductive"].unique()
)

print(
    "IsPeak:",
    dim_tariff_clean["IsPeak"].unique()
)

print(
    "IsPlanned:",
    dim_downtime_reason_clean["IsPlanned"].unique()
)

IsProductive: [0 1]
IsPeak: [0 1]
IsPlanned: [1 0]


In [46]:
dim_status_clean["IsProductive"] = (
    dim_status_clean["IsProductive"].astype("boolean")
)

dim_tariff_clean["IsPeak"] = (
    dim_tariff_clean["IsPeak"].astype("boolean")
)

dim_downtime_reason_clean["IsPlanned"] = (
    dim_downtime_reason_clean["IsPlanned"].astype("boolean")
)

### 4.7 Verify Standardized Data Types

Review the resulting schemas after type conversion to confirm that the transformations were applied successfully and no unexpected conversion issues were introduced.

In [47]:
fact_production_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2947392 entries, 0 to 2947391
Data columns (total 23 columns):
 #   Column                Dtype         
---  ------                -----         
 0   ReadingID             int64         
 1   Timestamp             datetime64[us]
 2   PlantID               str           
 3   ShiftID               str           
 4   TariffID              str           
 5   StatusID              Int64         
 6   MachineID             str           
 7   MachineName           str           
 8   MachineType           str           
 9   LineID                str           
 10  LineName              str           
 11  ProductID             str           
 12  ProductName           str           
 13  ProductFamily         str           
 14  StandardCycleTimeSec  Int64         
 15  ItemsProduced         Int64         
 16  RejectedItems         Int64         
 17  ActualCycleTimeSec    float64       
 18  PowerAvgKW            float64       
 19  PowerMaxKW 

In [48]:
fact_downtime_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 13046 entries, 0 to 13045
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   DowntimeEventID          13046 non-null  str           
 1   PlantID                  13046 non-null  str           
 2   MachineID                13046 non-null  str           
 3   LineID                   13046 non-null  str           
 4   ShiftID                  13046 non-null  str           
 5   StartTimestamp           13046 non-null  datetime64[us]
 6   EndTimestamp             13046 non-null  datetime64[us]
 7   DurationMinutes          13046 non-null  Int64         
 8   ReasonID                 13046 non-null  str           
 9   ProductID                13046 non-null  str           
 10  EnergyDuringDowntimeKWh  13046 non-null  float64       
dtypes: Int64(1), datetime64[us](2), float64(1), str(7)
memory usage: 1.6 MB


In [49]:
for name, df in {
    "Dim_Plant": dim_plant_clean,
    "Dim_Shift": dim_shift_clean,
    "Dim_Status": dim_status_clean,
    "Dim_EnergyTariff": dim_tariff_clean,
    "Dim_DowntimeReason": dim_downtime_reason_clean
}.items():

    print(f"\n{name}")
    print(df.dtypes)


Dim_Plant
PlantID         str
PlantName       str
City            str
Country         str
GridRegion      str
IndustryType    str
dtype: object

Dim_Shift
ShiftID      str
ShiftName    str
StartTime    str
EndTime      str
dtype: object

Dim_Status
StatusID           int64
StatusName           str
IsProductive     boolean
SeverityLevel        str
dtype: object

Dim_EnergyTariff
TariffID            str
TariffName          str
CalendarYear      int64
TimeBand            str
StartTime           str
EndTime             str
RatePerKWh      float64
IsPeak          boolean
dtype: object

Dim_DowntimeReason
ReasonID              str
ReasonCategory        str
ReasonName            str
IsPlanned         boolean
SeverityWeight      int64
dtype: object


In [50]:
errors="coerce"

### 4.8 Validate Conversion Results

Compare missing-value counts before and after type conversion to detect values that may have become missing because they could not be parsed into the expected data type.

In [51]:
conversion_check_columns = [
    "Timestamp",
    "StatusID",
    "StandardCycleTimeSec",
    "ItemsProduced",
    "RejectedItems",
    "ActualCycleTimeSec",
    "PowerAvgKW",
    "PowerMaxKW",
    "PowerMinKW",
    "EnergyKWh",
    "AmbientTempC"
]

conversion_validation = pd.DataFrame({
    "Before_Missing": fact_production[
        conversion_check_columns
    ].isna().sum(),

    "After_Missing": fact_production_clean[
        conversion_check_columns
    ].isna().sum()
})

conversion_validation["New_Missing_After_Conversion"] = (
    conversion_validation["After_Missing"]
    - conversion_validation["Before_Missing"]
)

conversion_validation

,Before_Missing,After_Missing,New_Missing_After_Conversion
Timestamp,0,0,0
StatusID,0,0,0
StandardCycleTimeSec,184906,184906,0
ItemsProduced,0,0,0
RejectedItems,0,0,0
ActualCycleTimeSec,230462,230462,0
PowerAvgKW,0,0,0
PowerMaxKW,0,0,0
PowerMinKW,0,0,0
EnergyKWh,0,0,0


# 5. Missing Value Assessment & Classification

## Objective
Evaluate missing values within their operational context and distinguish valid business nulls from potential data-quality issues.

Missing values will not be automatically removed or replaced. Each pattern will be assessed according to the operating status and business meaning of the affected fields.

### 5.1 Assess Product-Related Missing Values

Investigate missing product information and standard cycle time by operating status to determine whether these nulls represent expected non-production conditions or unexpected data-quality issues.

In [52]:
product_null_columns = [
    "ProductID",
    "ProductName",
    "ProductFamily",
    "StandardCycleTimeSec"
]

product_null_analysis = (
    fact_production_clean
    .groupby("StatusID", dropna=False)[product_null_columns]
    .agg(lambda x: x.isna().sum())
)

product_null_analysis

,ProductID,ProductName,ProductFamily,StandardCycleTimeSec
StatusID,,,,
0,184906,184906,184906,184906
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0


### 5.2 Assess Missing Actual Cycle Time

Evaluate missing actual cycle-time values by operating status to determine whether they occur in contexts where cycle time is not operationally applicable.

In [53]:
cycle_time_null_analysis = (
    fact_production_clean
    .groupby("StatusID", dropna=False)
    .agg(
        Total_Rows=("ReadingID", "size"),
        Missing_ActualCycleTime=(
            "ActualCycleTimeSec",
            lambda x: x.isna().sum()
        )
    )
)

cycle_time_null_analysis

,Total_Rows,Missing_ActualCycleTime
StatusID,,
0,184906,184906
1,74402,0
2,2642528,0
3,45556,45556


### 5.3 Identify Unexpected Nulls During Production

Check whether production-related fields are missing while the machine is operating in a productive status.

Missing operational values during productive states may represent genuine data-quality issues and require further investigation.

In [54]:
productive_status_ids = dim_status_clean.loc[
    dim_status_clean["IsProductive"],
    "StatusID"
].tolist()

productive_status_ids

[1, 2]

In [55]:
production_rows = fact_production_clean[
    fact_production_clean["StatusID"].isin(productive_status_ids)
]

In [56]:
required_during_production = [
    "ProductID",
    "ProductName",
    "ProductFamily",
    "StandardCycleTimeSec",
    "ActualCycleTimeSec"
]

unexpected_production_nulls = (
    production_rows[required_during_production]
    .isna()
    .sum()
    .to_frame("Unexpected_Missing_Count")
)

unexpected_production_nulls

,Unexpected_Missing_Count
ProductID,0
ProductName,0
ProductFamily,0
StandardCycleTimeSec,0
ActualCycleTimeSec,0


### 5.4 Missing Value Classification

Classify the observed missing-value patterns based on their operational meaning and validation results.

The purpose of this classification is to preserve legitimate non-applicable values while isolating unexpected missing data for further investigation.

In [57]:
missing_classification = pd.DataFrame({
    "Fields": [
        "ProductID / ProductName / ProductFamily / StandardCycleTimeSec",
        "ActualCycleTimeSec"
    ],
    "Expected_Context": [
        "Idle",
        "Idle and Alarm/Interrupted"
    ],
    "Classification": [
        "Business Null",
        "Business Null"
    ],
    "Treatment": [
        "Keep as missing",
        "Keep as missing"
    ]
})

missing_classification

,Fields,Expected_Context,Classification,Treatment
0,ProductID / ProductName / ProductFamily / Stan...,Idle,Business Null,Keep as missing
1,ActualCycleTimeSec,Idle and Alarm/Interrupted,Business Null,Keep as missing


### 5.5 Validate Critical Operational Fields

Check essential operational fields for unexpected missing values.

These fields are expected to be available for every production-energy observation regardless of whether the machine is productive, idle, or interrupted.

In [58]:
critical_columns = [
    "ReadingID",
    "Timestamp",
    "PlantID",
    "ShiftID",
    "TariffID",
    "StatusID",
    "MachineID",
    "LineID",
    "PowerAvgKW",
    "PowerMaxKW",
    "PowerMinKW",
    "EnergyKWh"
]

critical_missing = (
    fact_production_clean[
        critical_columns
    ]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

critical_missing

,Missing_Count
ReadingID,0
Timestamp,0
PlantID,0
ShiftID,0
TariffID,0
StatusID,0
MachineID,0
LineID,0
PowerAvgKW,0
PowerMaxKW,0


### 5.6 Assess Missing Values in Downtime Events

Evaluate missing values in the downtime-event dataset and distinguish optional contextual attributes from fields required to define a valid downtime event.

In [59]:
downtime_missing_profile = pd.DataFrame({
    "Missing_Count": fact_downtime_clean.isna().sum(),
    "Missing_Percent": (
        fact_downtime_clean.isna().mean() * 100
    ).round(2)
})

downtime_missing_profile[
    downtime_missing_profile["Missing_Count"] > 0
].sort_values(
    "Missing_Percent",
    ascending=False
)

,Missing_Count,Missing_Percent


### 5.7 Validate Critical Downtime Fields

Check whether essential attributes required to define and analyze a downtime event contain missing values.

In [60]:
critical_downtime_columns = [
    "DowntimeEventID",
    "PlantID",
    "MachineID",
    "LineID",
    "ShiftID",
    "StartTimestamp",
    "EndTimestamp",
    "DurationMinutes",
    "ReasonID",
    "EnergyDuringDowntimeKWh"
]

downtime_critical_missing = (
    fact_downtime_clean[
        critical_downtime_columns
    ]
    .isna()
    .sum()
    .to_frame("Missing_Count")
)

downtime_critical_missing

,Missing_Count
DowntimeEventID,0
PlantID,0
MachineID,0
LineID,0
ShiftID,0
StartTimestamp,0
EndTimestamp,0
DurationMinutes,0
ReasonID,0
EnergyDuringDowntimeKWh,0


### 5.8 Missing Value Classification Summary

Summarize the interpretation and treatment of the main missing-value patterns identified in the datasets.

Valid business nulls are preserved as missing values because replacing them with artificial values such as zero would change their operational meaning.

In [61]:
missing_classification = pd.DataFrame({
    "Field_Group": [
        "Product attributes",
        "StandardCycleTimeSec",
        "ActualCycleTimeSec"
    ],
    "Expected_Missing_Context": [
        "Idle",
        "Idle",
        "Idle and Alarm/Interrupted"
    ],
    "Classification": [
        "Business Null",
        "Business Null",
        "Business Null"
    ],
    "Treatment": [
        "Keep as missing",
        "Keep as missing",
        "Keep as missing"
    ]
})

missing_classification

,Field_Group,Expected_Missing_Context,Classification,Treatment
0,Product attributes,Idle,Business Null,Keep as missing
1,StandardCycleTimeSec,Idle,Business Null,Keep as missing
2,ActualCycleTimeSec,Idle and Alarm/Interrupted,Business Null,Keep as missing


### Missing Value Decision

The observed missing values are retained when they represent non-applicable operational conditions rather than incomplete data.

In particular, product-related attributes are not applicable during idle operation, while actual cycle time is not applicable during idle and interrupted states. These values are therefore preserved as null rather than replaced with zero.

# 6. Duplicate & Key Validation

## Objective
Validate record uniqueness and primary-key integrity across all fact and dimension tables.

This stage distinguishes between:
- Exact duplicate rows
- Duplicate primary-key values
- Missing key values

No duplicate records will be removed automatically before their cause is understood.ذذ

### 6.1 Exact Duplicate Check

Check all datasets for fully duplicated rows where every column contains the same values.

Exact duplicates are identified separately from duplicated identifiers because the two conditions may represent different data-quality issues.

In [62]:
duplicate_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Exact_Duplicate_Rows": df.duplicated().sum()
    }
    for name, df in {
        "Fact_ProductionEnergy": fact_production_clean,
        "Fact_DowntimeEvents": fact_downtime_clean,
        "Dim_Plant": dim_plant_clean,
        "Dim_Shift": dim_shift_clean,
        "Dim_Status": dim_status_clean,
        "Dim_EnergyTariff": dim_tariff_clean,
        "Dim_DowntimeReason": dim_downtime_reason_clean
    }.items()
])

duplicate_summary

,Dataset,Exact_Duplicate_Rows
0,Fact_ProductionEnergy,0
1,Fact_DowntimeEvents,0
2,Dim_Plant,0
3,Dim_Shift,0
4,Dim_Status,0
5,Dim_EnergyTariff,0
6,Dim_DowntimeReason,0


### 6.2 Define Expected Primary Keys

Define the identifier expected to uniquely represent each record in every fact and dimension table.

In [63]:
primary_keys = {
    "Fact_ProductionEnergy": "ReadingID",
    "Fact_DowntimeEvents": "DowntimeEventID",
    "Dim_Plant": "PlantID",
    "Dim_Shift": "ShiftID",
    "Dim_Status": "StatusID",
    "Dim_EnergyTariff": "TariffID",
    "Dim_DowntimeReason": "ReasonID"
}

### 6.3 Primary-Key Integrity Check

Validate that each expected primary key:
1. Contains no missing values.
2. Contains no duplicate values.
3. Uniquely identifies every record in its table.

In [64]:
clean_datasets = {
    "Fact_ProductionEnergy": fact_production_clean,
    "Fact_DowntimeEvents": fact_downtime_clean,
    "Dim_Plant": dim_plant_clean,
    "Dim_Shift": dim_shift_clean,
    "Dim_Status": dim_status_clean,
    "Dim_EnergyTariff": dim_tariff_clean,
    "Dim_DowntimeReason": dim_downtime_reason_clean
}

In [65]:
key_results = []

for table_name, key_column in primary_keys.items():

    df = clean_datasets[table_name]

    key_results.append({
        "Dataset": table_name,
        "Primary_Key": key_column,
        "Rows": len(df),
        "Missing_Keys": df[key_column].isna().sum(),
        "Duplicate_Keys": df[key_column].duplicated().sum(),
        "Unique_Keys": df[key_column].nunique(dropna=True)
    })

key_validation = pd.DataFrame(key_results)

key_validation

,Dataset,Primary_Key,Rows,Missing_Keys,Duplicate_Keys,Unique_Keys
0,Fact_ProductionEnergy,ReadingID,2947392,0,0,2947392
1,Fact_DowntimeEvents,DowntimeEventID,13046,0,0,13046
2,Dim_Plant,PlantID,2,0,0,2
3,Dim_Shift,ShiftID,3,0,0,3
4,Dim_Status,StatusID,4,0,0,4
5,Dim_EnergyTariff,TariffID,8,0,0,8
6,Dim_DowntimeReason,ReasonID,6,0,0,6


### 6.4 Inspect Duplicate Key Records

If duplicate primary-key values are detected, retrieve the affected records for investigation before deciding whether any record should be removed.

In [66]:
duplicate_reading_ids = fact_production_clean[
    fact_production_clean["ReadingID"].duplicated(keep=False)
].sort_values("ReadingID")

duplicate_reading_ids.head()

,ReadingID,Timestamp,PlantID,ShiftID,TariffID,StatusID,MachineID,MachineName,MachineType,LineID,LineName,ProductID,ProductName,ProductFamily,StandardCycleTimeSec,ItemsProduced,RejectedItems,ActualCycleTimeSec,PowerAvgKW,PowerMaxKW,PowerMinKW,EnergyKWh,AmbientTempC


In [67]:
duplicate_downtime_ids = fact_downtime_clean[
    fact_downtime_clean["DowntimeEventID"].duplicated(keep=False)
].sort_values("DowntimeEventID")

duplicate_downtime_ids.head()

,DowntimeEventID,PlantID,MachineID,LineID,ShiftID,StartTimestamp,EndTimestamp,DurationMinutes,ReasonID,ProductID,EnergyDuringDowntimeKWh


### Duplicate and Key Validation Decision

No records are removed solely based on duplicate screening.

Primary-key and exact-duplicate checks are used to confirm record integrity. Any detected duplicate would require investigation before removal to avoid eliminating valid operational observations.

The validation results showed no missing or duplicated primary-key values in the assessed tables, confirming that the expected identifiers uniquely represent their records.

# 7. Business Rule Validation

## Objective
Validate whether operational values and relationships follow the expected manufacturing logic.

This stage identifies:
- Impossible values
- Logically inconsistent records
- Potential measurement issues
- Operational observations requiring investigation

Validation failures are flagged before any correction or exclusion decision is made.

### 7.1 Validate Production and Reject Quantities

Verify that production and rejection quantities follow basic manufacturing constraints:

- Items produced cannot be negative.
- Rejected items cannot be negative.
- Rejected items cannot exceed total items produced.

In [68]:
production_quantity_validation = pd.DataFrame({
    "Rule": [
        "ItemsProduced >= 0",
        "RejectedItems >= 0",
        "RejectedItems <= ItemsProduced"
    ],

    "Failed_Rows": [
        (fact_production_clean["ItemsProduced"] < 0).sum(),

        (fact_production_clean["RejectedItems"] < 0).sum(),

        (
            fact_production_clean["RejectedItems"]
            > fact_production_clean["ItemsProduced"]
        ).sum()
    ]
})

production_quantity_validation

,Rule,Failed_Rows
0,ItemsProduced >= 0,0
1,RejectedItems >= 0,0
2,RejectedItems <= ItemsProduced,0


### 7.2 Validate Energy and Power Measurements

Check whether energy and power measurements contain impossible negative values.

Negative consumption values are treated as potential data-quality issues unless a valid business explanation exists.

In [69]:
energy_power_validation = pd.DataFrame({
    "Rule": [
        "EnergyKWh >= 0",
        "PowerAvgKW >= 0",
        "PowerMaxKW >= 0",
        "PowerMinKW >= 0"
    ],

    "Failed_Rows": [
        (fact_production_clean["EnergyKWh"] < 0).sum(),
        (fact_production_clean["PowerAvgKW"] < 0).sum(),
        (fact_production_clean["PowerMaxKW"] < 0).sum(),
        (fact_production_clean["PowerMinKW"] < 0).sum()
    ]
})

energy_power_validation

,Rule,Failed_Rows
0,EnergyKWh >= 0,0
1,PowerAvgKW >= 0,0
2,PowerMaxKW >= 0,0
3,PowerMinKW >= 0,0


### 7.3 Validate Power Measurement Consistency

Verify the expected relationship between minimum, average, and maximum power measurements.

For a valid reading:

`PowerMinKW <= PowerAvgKW <= PowerMaxKW`ذ

In [70]:
invalid_power_order = ~(
    (fact_production_clean["PowerMinKW"]
     <= fact_production_clean["PowerAvgKW"])
    &
    (fact_production_clean["PowerAvgKW"]
     <= fact_production_clean["PowerMaxKW"])
)

invalid_power_order.sum()

np.int64(0)

### 7.4 Validate Cycle-Time Measurements

Verify that cycle-time values are positive when they are operationally applicable.

Zero or negative cycle times during productive operation may indicate invalid measurements or data-recording issues.

In [71]:
invalid_standard_cycle = (
    fact_production_clean["StandardCycleTimeSec"].notna()
    &
    (fact_production_clean["StandardCycleTimeSec"] <= 0)
)

invalid_standard_cycle.sum()

np.int64(0)

In [72]:
invalid_actual_cycle = (
    fact_production_clean["StatusID"].isin(productive_status_ids)
    &
    (
        fact_production_clean["ActualCycleTimeSec"].isna()
        |
        (fact_production_clean["ActualCycleTimeSec"] <= 0)
    )
)

invalid_actual_cycle.sum()

np.int64(0)

### 7.5 Validate Idle-State Production Logic

Verify that idle observations do not report produced or rejected units.

Energy consumption is not required to be zero during idle periods because machines may continue consuming standby or auxiliary energy.

In [73]:
idle_status_ids = (
    dim_status_clean
    .loc[
        dim_status_clean["StatusName"]
        .str.contains("Idle", case=False, na=False),
        "StatusID"
    ]
    .tolist()
)

idle_status_ids

[0]

In [74]:
idle_rows = fact_production_clean[
    fact_production_clean["StatusID"].isin(idle_status_ids)
]

In [75]:
idle_production_validation = pd.DataFrame({
    "Rule": [
        "Idle rows with ItemsProduced > 0",
        "Idle rows with RejectedItems > 0"
    ],

    "Failed_Rows": [
        (idle_rows["ItemsProduced"] > 0).sum(),
        (idle_rows["RejectedItems"] > 0).sum()
    ]
})

idle_production_validation

,Rule,Failed_Rows
0,Idle rows with ItemsProduced > 0,0
1,Idle rows with RejectedItems > 0,0


### 7.6 Assess Productive States with Zero Output

Identify productive-status observations where no units were recorded.

These records are treated as operational warnings rather than automatically classified as errors because short production intervals may legitimately contain zero completed units.

In [76]:
productive_zero_output = (
    fact_production_clean["StatusID"].isin(productive_status_ids)
    &
    (fact_production_clean["ItemsProduced"] == 0)
)

productive_zero_output.sum()

np.int64(0)

In [77]:
fact_production_clean.loc[
    productive_zero_output,
    [
        "ReadingID",
        "Timestamp",
        "MachineID",
        "ProductID",
        "StatusID",
        "ItemsProduced",
        "ActualCycleTimeSec",
        "EnergyKWh"
    ]
].head()

,ReadingID,Timestamp,MachineID,ProductID,StatusID,ItemsProduced,ActualCycleTimeSec,EnergyKWh


### 7.7 Validate Energy–Power Consistency

Validate the relationship between average power and recorded energy consumption.

Because each production-energy observation represents a five-minute interval:

`Expected Energy (kWh) = PowerAvgKW × (5 / 60)`

or equivalently:

`Expected Energy (kWh) = PowerAvgKW / 12`

A tolerance-based comparison is used because small differences may result from rounding or measurement precision.

In [78]:
expected_energy_check = (
    fact_production_clean["PowerAvgKW"] / 12
)

energy_difference = (
    fact_production_clean["EnergyKWh"]
    - expected_energy_check
)

energy_abs_error_pct = (
    energy_difference.abs()
    / expected_energy_check.replace(0, np.nan)
    * 100
)

In [79]:
energy_validation_summary = pd.Series({
    "Median_Error_Pct": energy_abs_error_pct.median(),
    "Mean_Error_Pct": energy_abs_error_pct.mean(),
    "95th_Percentile_Error_Pct": energy_abs_error_pct.quantile(0.95),
    "Maximum_Error_Pct": energy_abs_error_pct.max()
})

energy_validation_summary

Median_Error_Pct            0.00
Mean_Error_Pct              0.00
95th_Percentile_Error_Pct   0.01
Maximum_Error_Pct           0.06
dtype: float64

### 7.8 Validate Downtime Duration

Compare the recorded downtime duration with the duration calculated from event start and end timestamps.

For each event:

`Calculated Duration = EndTimestamp - StartTimestamp`

In [80]:
calculated_duration_minutes = (
    (
        fact_downtime_clean["EndTimestamp"]
        - fact_downtime_clean["StartTimestamp"]
    )
    .dt.total_seconds()
    / 60
)

In [81]:
downtime_duration_difference = (
    fact_downtime_clean["DurationMinutes"]
    - calculated_duration_minutes
)

downtime_duration_difference.describe()

count   13,046.00
mean         0.00
std          0.00
min          0.00
25%          0.00
50%          0.00
75%          0.00
max          0.00
dtype: Float64

In [82]:
downtime_duration_mismatch = (
    downtime_duration_difference.abs() > 0.01
)

downtime_duration_mismatch.sum()

np.int64(0)

In [83]:
fact_downtime_clean.loc[
    downtime_duration_mismatch,
    [
        "DowntimeEventID",
        "StartTimestamp",
        "EndTimestamp",
        "DurationMinutes"
    ]
].head(20)

,DowntimeEventID,StartTimestamp,EndTimestamp,DurationMinutes


### 7.9 Validate Downtime Event Time Order

Verify that every downtime event ends after it starts and has a positive recorded duration.

In [84]:
downtime_time_validation = pd.DataFrame({
    "Rule": [
        "EndTimestamp >= StartTimestamp",
        "DurationMinutes > 0"
    ],

    "Failed_Rows": [
        (
            fact_downtime_clean["EndTimestamp"]
            < fact_downtime_clean["StartTimestamp"]
        ).sum(),

        (
            fact_downtime_clean["DurationMinutes"] <= 0
        ).sum()
    ]
})

downtime_time_validation

,Rule,Failed_Rows
0,EndTimestamp >= StartTimestamp,0
1,DurationMinutes > 0,0


### 7.10 Business Rule Validation Summary

Consolidate the main validation results into a single quality-control summary.

The summary distinguishes hard logical violations from observations that may require operational investigation rather than automatic correction.

In [85]:
business_rule_summary = pd.DataFrame({
    "Validation_Rule": [
        "ItemsProduced >= 0",
        "RejectedItems >= 0",
        "RejectedItems <= ItemsProduced",
        "EnergyKWh >= 0",
        "PowerAvgKW >= 0",
        "PowerMaxKW >= 0",
        "PowerMinKW >= 0",
        "PowerMinKW <= PowerAvgKW <= PowerMaxKW",
        "StandardCycleTimeSec > 0 when available",
        "ActualCycleTimeSec > 0 during production",
        "Idle ItemsProduced = 0",
        "Idle RejectedItems = 0",
        "Productive status with zero output",
        "Downtime EndTimestamp >= StartTimestamp",
        "Downtime DurationMinutes > 0",
        "Recorded downtime duration matches timestamps"
    ],

    "Failed_Rows": [
        (fact_production_clean["ItemsProduced"] < 0).sum(),
        (fact_production_clean["RejectedItems"] < 0).sum(),
        (
            fact_production_clean["RejectedItems"]
            > fact_production_clean["ItemsProduced"]
        ).sum(),

        (fact_production_clean["EnergyKWh"] < 0).sum(),
        (fact_production_clean["PowerAvgKW"] < 0).sum(),
        (fact_production_clean["PowerMaxKW"] < 0).sum(),
        (fact_production_clean["PowerMinKW"] < 0).sum(),

        invalid_power_order.sum(),
        invalid_standard_cycle.sum(),
        invalid_actual_cycle.sum(),

        (idle_rows["ItemsProduced"] > 0).sum(),
        (idle_rows["RejectedItems"] > 0).sum(),

        productive_zero_output.sum(),

        (
            fact_downtime_clean["EndTimestamp"]
            < fact_downtime_clean["StartTimestamp"]
        ).sum(),

        (
            fact_downtime_clean["DurationMinutes"] <= 0
        ).sum(),

        downtime_duration_mismatch.sum()
    ]
})

business_rule_summary

,Validation_Rule,Failed_Rows
0,ItemsProduced >= 0,0
1,RejectedItems >= 0,0
2,RejectedItems <= ItemsProduced,0
3,EnergyKWh >= 0,0
4,PowerAvgKW >= 0,0
5,PowerMaxKW >= 0,0
6,PowerMinKW >= 0,0
7,PowerMinKW <= PowerAvgKW <= PowerMaxKW,0
8,StandardCycleTimeSec > 0 when available,0
9,ActualCycleTimeSec > 0 during production,0


### Validation Decision Principle

Validation failures are not automatically removed.

Each identified issue will be classified as one of the following:

- **Valid:** The observation follows the expected data and business rules.
- **Business Null:** The missing value is logically not applicable.
- **Data Error:** The record violates a hard structural or logical rule.
- **Data Warning:** The record is unusual and requires further review.
- **Operational Anomaly:** The value is valid but represents potentially abnormal manufacturing behavior that should remain available for analysis.

This distinction prevents meaningful operational inefficiencies from being incorrectly removed during data cleaning.

# 8. Time-Series Integrity Validation

## Objective
Validate the temporal consistency of the manufacturing data before time-based analysis.

This stage checks:
- Duplicate machine-timestamp combinations
- Chronological consistency
- Expected five-minute observation intervals
- Missing time intervals
- Shift assignment consistency
- Timestamp coverage by machine

> **Note:** Missing intervals are flagged for investigation and are not automatically treated as data errors, since they may reflect downtime, data collection gaps, or operational conditions.

# 8. Time-Series Integrity Validation

## Objective
Validate the temporal consistency of the manufacturing data before time-based analysis.

This stage checks:
- Duplicate machine-timestamp combinations
- Chronological consistency
- Expected five-minute observation intervals
- Missing time intervals
- Shift assignment consistency
- Timestamp coverage by machine

> **Note:** Missing intervals are flagged for investigation and are not automatically treated as data errors, since they may reflect downtime, data collection gaps, or operational conditions.

In [86]:
machine_timestamp_duplicates = (
    fact_production_clean
    .duplicated(
        subset=["MachineID", "Timestamp"],
        keep=False
    )
)

machine_timestamp_duplicates.sum()

np.int64(0)

In [87]:
fact_production_clean.loc[
    machine_timestamp_duplicates,
    [
        "ReadingID",
        "Timestamp",
        "PlantID",
        "MachineID",
        "StatusID",
        "ProductID",
        "EnergyKWh"
    ]
].sort_values(
    ["MachineID", "Timestamp"]
).head(20)

,ReadingID,Timestamp,PlantID,MachineID,StatusID,ProductID,EnergyKWh


### 8.2 Machine-Level Temporal Coverage

Review the first and last available timestamp for each machine to confirm its observed operating period and identify differences in coverage across assets.

In [88]:
machine_time_coverage = (
    fact_production_clean
    .groupby("MachineID")
    .agg(
        First_Timestamp=("Timestamp", "min"),
        Last_Timestamp=("Timestamp", "max"),
        Total_Readings=("ReadingID", "size")
    )
    .reset_index()
)

machine_time_coverage

,MachineID,First_Timestamp,Last_Timestamp,Total_Readings
0,MCH-001,2024-01-01,2025-12-31 23:55:00,210528
1,MCH-002,2024-01-01,2025-12-31 23:55:00,210528
2,MCH-003,2024-01-01,2025-12-31 23:55:00,210528
3,MCH-004,2024-01-01,2025-12-31 23:55:00,210528
4,MCH-005,2024-01-01,2025-12-31 23:55:00,210528
5,MCH-006,2024-01-01,2025-12-31 23:55:00,210528
6,MCH-007,2024-01-01,2025-12-31 23:55:00,210528
7,MCH-008,2024-01-01,2025-12-31 23:55:00,210528
8,MCH-009,2024-01-01,2025-12-31 23:55:00,210528
9,MCH-010,2024-01-01,2025-12-31 23:55:00,210528


### 8.3 Observation Interval Validation

Calculate the time difference between consecutive readings for each machine.

The expected regular interval is five minutes. Deviations from this interval are flagged for further assessment.

In [89]:
time_check = (
    fact_production_clean[
        ["MachineID", "Timestamp"]
    ]
    .sort_values(
        ["MachineID", "Timestamp"]
    )
    .copy()
)

In [90]:
time_check["Interval_Minutes"] = (
    time_check
    .groupby("MachineID")["Timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

### 8.4 Observation Interval Distribution

Summarize the calculated time intervals to determine how consistently the expected five-minute reading frequency is maintained.

In [91]:
time_check["Interval_Minutes"].value_counts().sort_index().head(20)

Interval_Minutes
5.00    2947378
Name: count, dtype: int64

### 8.5 Identify Time Gaps

Flag consecutive readings separated by more than the expected five-minute interval.

Time gaps are retained for investigation because they may indicate missing sensor readings, communication interruptions, or legitimate periods without recorded observations.

In [92]:
time_gaps = time_check[
    time_check["Interval_Minutes"] > 5
].copy()

print("Number of time gaps:", len(time_gaps))

time_gaps.head()

Number of time gaps: 0


,MachineID,Timestamp,Interval_Minutes


### 8.6 Estimate Missing Observation Intervals

Estimate the number of expected five-minute observations potentially missing within detected time gaps.

In [93]:
time_gaps["Estimated_Missing_Readings"] = (
    (time_gaps["Interval_Minutes"] / 5) - 1
).round().astype("Int64")

In [94]:
time_gap_summary = pd.Series({
    "Number_of_Gaps": len(time_gaps),
    "Estimated_Missing_Readings": (
        time_gaps["Estimated_Missing_Readings"].sum()
    ),
    "Largest_Gap_Minutes": (
        time_gaps["Interval_Minutes"].max()
    )
})

time_gap_summary

Number_of_Gaps               0.00
Estimated_Missing_Readings   0.00
Largest_Gap_Minutes           NaN
dtype: float64

### 8.7 Time Gaps by Machine

Summarize temporal gaps by machine to identify assets with unusually incomplete or interrupted time-series coverage.

In [95]:
machine_gap_summary = (
    time_gaps
    .groupby("MachineID")
    .agg(
        Gap_Count=("Interval_Minutes", "size"),
        Estimated_Missing_Readings=(
            "Estimated_Missing_Readings",
            "sum"
        ),
        Maximum_Gap_Minutes=(
            "Interval_Minutes",
            "max"
        )
    )
    .reset_index()
    .sort_values(
        "Estimated_Missing_Readings",
        ascending=False
    )
)

machine_gap_summary

,MachineID,Gap_Count,Estimated_Missing_Readings,Maximum_Gap_Minutes


### 8.8 Shift Assignment Validation

Validate whether the recorded ShiftID is consistent with the observation timestamp and the shift schedule defined in the shift dimension.

This ensures that shift-based performance comparisons use correctly assigned operational periods.

In [96]:
display(dim_shift_clean)

,ShiftID,ShiftName,StartTime,EndTime
0,S1,Morning,06:00,14:00
1,S2,Evening,14:00,22:00
2,S3,Night,22:00,06:00


In [97]:
dim_shift_clean.columns.tolist()

['ShiftID', 'ShiftName', 'StartTime', 'EndTime']

### 8.9 Prepare Shift Time Boundaries

Convert shift start and end values into comparable time representations before validating timestamp-to-shift assignments.

In [98]:
dim_shift_clean["StartTimeParsed"] = pd.to_datetime(
    dim_shift_clean["StartTime"],
    format="%H:%M",
    errors="coerce"
).dt.time

dim_shift_clean["EndTimeParsed"] = pd.to_datetime(
    dim_shift_clean["EndTime"],
    format="%H:%M",
    errors="coerce"
).dt.time

### 8.10 Derive Expected Shift from Timestamp

Derive the expected ShiftID from each observation timestamp using the shift boundaries defined in the reference table.

In [99]:
shift_rules = dim_shift_clean[
    ["ShiftID", "StartTimeParsed", "EndTimeParsed"]
].to_dict("records")

In [100]:
def get_expected_shift(timestamp):

    current_time = timestamp.time()

    for rule in shift_rules:

        start = rule["StartTimeParsed"]
        end = rule["EndTimeParsed"]

        if start < end:
            if start <= current_time < end:
                return rule["ShiftID"]

        else:
            if current_time >= start or current_time < end:
                return rule["ShiftID"]

    return pd.NA

In [101]:
fact_production_clean["ExpectedShiftID"] = (
    fact_production_clean["Timestamp"]
    .apply(get_expected_shift)
)

In [102]:
shift_mismatch = (
    fact_production_clean["ShiftID"]
    != fact_production_clean["ExpectedShiftID"]
)

shift_mismatch.sum()

np.int64(0)

In [103]:
fact_production_clean.loc[
    shift_mismatch,
    [
        "ReadingID",
        "Timestamp",
        "ShiftID",
        "ExpectedShiftID",
        "MachineID"
    ]
].head(20)

,ReadingID,Timestamp,ShiftID,ExpectedShiftID,MachineID


### 8.11 Time-Series Validation Summary

Consolidate the main time-series integrity checks into a single validation summary.

In [104]:
time_series_summary = pd.DataFrame({
    "Validation_Check": [
        "Duplicate Machine-Timestamp Records",
        "Intervals Greater Than 5 Minutes",
        "Estimated Missing Readings",
        "Shift Assignment Mismatches"
    ],

    "Affected_Records": [
        machine_timestamp_duplicates.sum(),
        len(time_gaps),
        time_gaps["Estimated_Missing_Readings"].sum(),
        shift_mismatch.sum()
    ]
})

time_series_summary

,Validation_Check,Affected_Records
0,Duplicate Machine-Timestamp Records,0
1,Intervals Greater Than 5 Minutes,0
2,Estimated Missing Readings,0
3,Shift Assignment Mismatches,0


# 9. Referential Integrity Validation

## Objective
Validate relationships between fact and dimension tables by ensuring that foreign-key values in the fact tables correspond to valid records in the relevant dimensions.

This stage helps identify orphan records, invalid identifiers, and inconsistencies that could affect SQL relationships and Power BI modeling.

### 9.1 Validate Plant References

Verify that every PlantID used in the fact tables exists in the plant dimension.

In [105]:
invalid_production_plants = (
    ~fact_production_clean["PlantID"]
    .isin(dim_plant_clean["PlantID"])
)

invalid_production_plants.sum()

np.int64(0)

In [106]:
invalid_downtime_plants = (
    ~fact_downtime_clean["PlantID"]
    .isin(dim_plant_clean["PlantID"])
)

invalid_downtime_plants.sum()

np.int64(0)

### 9.2 Validate Shift References

Verify that all ShiftID values used in the fact tables exist in the shift dimension.

In [107]:
invalid_production_shifts = (
    ~fact_production_clean["ShiftID"]
    .isin(dim_shift_clean["ShiftID"])
)

invalid_downtime_shifts = (
    ~fact_downtime_clean["ShiftID"]
    .isin(dim_shift_clean["ShiftID"])
)

print(
    "Invalid production ShiftIDs:",
    invalid_production_shifts.sum()
)

print(
    "Invalid downtime ShiftIDs:",
    invalid_downtime_shifts.sum()
)

Invalid production ShiftIDs: 0
Invalid downtime ShiftIDs: 0


### 9.3 Validate Status References

Verify that every StatusID recorded in the production-energy fact table exists in the status dimension.ة

In [108]:
invalid_status = (
    ~fact_production_clean["StatusID"]
    .isin(dim_status_clean["StatusID"])
)

invalid_status.sum()

np.int64(0)

### 9.4 Validate Tariff References

Verify that every TariffID recorded in the production-energy fact table corresponds to a valid tariff record.

In [109]:
invalid_tariff = (
    ~fact_production_clean["TariffID"]
    .isin(dim_tariff_clean["TariffID"])
)

invalid_tariff.sum()

np.int64(0)

### 9.5 Validate Downtime Reason References

Verify that each ReasonID recorded in the downtime fact table exists in the downtime-reason dimension.

In [110]:
invalid_downtime_reason = (
    ~fact_downtime_clean["ReasonID"]
    .isin(dim_downtime_reason_clean["ReasonID"])
)

invalid_downtime_reason.sum()

np.int64(0)

### 9.6 Validate Product Attribute Consistency

Check whether each ProductID is consistently associated with a single product name, product family, and standard cycle time.

This validation is performed before extracting the product dimension during normalization.

In [111]:
product_consistency = (
    fact_production_clean[
        fact_production_clean["ProductID"].notna()
    ]
    .groupby("ProductID")
    .agg(
        ProductName_Count=("ProductName", "nunique"),
        ProductFamily_Count=("ProductFamily", "nunique"),
        StandardCycleTime_Count=(
            "StandardCycleTimeSec",
            "nunique"
        )
    )
    .reset_index()
)

product_consistency

,ProductID,ProductName_Count,ProductFamily_Count,StandardCycleTime_Count
0,PRD-001,1,1,1
1,PRD-002,1,1,1
2,PRD-003,1,1,1
3,PRD-004,1,1,1
4,PRD-005,1,1,1
5,PRD-006,1,1,1
6,PRD-007,1,1,1
7,PRD-008,1,1,1
8,PRD-009,1,1,1
9,PRD-010,1,1,1


In [112]:
invalid_product_consistency = product_consistency[
    (product_consistency["ProductName_Count"] > 1)
    |
    (product_consistency["ProductFamily_Count"] > 1)
    |
    (product_consistency["StandardCycleTime_Count"] > 1)
]

invalid_product_consistency

,ProductID,ProductName_Count,ProductFamily_Count,StandardCycleTime_Count


### 9.7 Validate Machine Attribute Consistency

Check whether each MachineID consistently maps to a single machine name, machine type, line, and plant before creating the machine dimension.

In [113]:
machine_consistency = (
    fact_production_clean
    .groupby("MachineID")
    .agg(
        MachineName_Count=("MachineName", "nunique"),
        MachineType_Count=("MachineType", "nunique"),
        LineID_Count=("LineID", "nunique"),
        PlantID_Count=("PlantID", "nunique")
    )
    .reset_index()
)

machine_consistency

,MachineID,MachineName_Count,MachineType_Count,LineID_Count,PlantID_Count
0,MCH-001,1,1,1,1
1,MCH-002,1,1,1,1
2,MCH-003,1,1,1,1
3,MCH-004,1,1,1,1
4,MCH-005,1,1,1,1
5,MCH-006,1,1,1,1
6,MCH-007,1,1,1,1
7,MCH-008,1,1,1,1
8,MCH-009,1,1,1,1
9,MCH-010,1,1,1,1


In [114]:
invalid_machine_consistency = machine_consistency[
    (machine_consistency["MachineName_Count"] > 1)
    |
    (machine_consistency["MachineType_Count"] > 1)
    |
    (machine_consistency["LineID_Count"] > 1)
    |
    (machine_consistency["PlantID_Count"] > 1)
]

invalid_machine_consistency

,MachineID,MachineName_Count,MachineType_Count,LineID_Count,PlantID_Count


### 9.8 Validate Line Attribute Consistency

Verify that each LineID is consistently associated with a single line name and plant before creating the line dimension.

In [115]:
line_consistency = (
    fact_production_clean
    .groupby("LineID")
    .agg(
        LineName_Count=("LineName", "nunique"),
        PlantID_Count=("PlantID", "nunique")
    )
    .reset_index()
)

line_consistency

,LineID,LineName_Count,PlantID_Count
0,LN-01,1,1
1,LN-02,1,1
2,LN-03,1,1
3,LN-04,1,1
4,LN-05,1,1


In [116]:
invalid_line_consistency = line_consistency[
    (line_consistency["LineName_Count"] > 1)
    |
    (line_consistency["PlantID_Count"] > 1)
]

invalid_line_consistency

,LineID,LineName_Count,PlantID_Count


### 9.9 Referential Integrity Summary

Consolidate foreign-key and entity-consistency validation results before building the normalized analytical model.

In [117]:
referential_integrity_summary = pd.DataFrame({
    "Validation_Check": [
        "Invalid Production PlantID",
        "Invalid Downtime PlantID",
        "Invalid Production ShiftID",
        "Invalid Downtime ShiftID",
        "Invalid StatusID",
        "Invalid TariffID",
        "Invalid Downtime ReasonID",
        "Inconsistent ProductIDs",
        "Inconsistent MachineIDs",
        "Inconsistent LineIDs"
    ],

    "Affected_Records_or_Entities": [
        invalid_production_plants.sum(),
        invalid_downtime_plants.sum(),
        invalid_production_shifts.sum(),
        invalid_downtime_shifts.sum(),
        invalid_status.sum(),
        invalid_tariff.sum(),
        invalid_downtime_reason.sum(),
        len(invalid_product_consistency),
        len(invalid_machine_consistency),
        len(invalid_line_consistency)
    ]
})

referential_integrity_summary

,Validation_Check,Affected_Records_or_Entities
0,Invalid Production PlantID,0
1,Invalid Downtime PlantID,0
2,Invalid Production ShiftID,0
3,Invalid Downtime ShiftID,0
4,Invalid StatusID,0
5,Invalid TariffID,0
6,Invalid Downtime ReasonID,0
7,Inconsistent ProductIDs,0
8,Inconsistent MachineIDs,0
9,Inconsistent LineIDs,0


### Referential Integrity Decision

Foreign-key and entity-consistency issues are reviewed before normalization.

Valid relationships are preserved, while orphan identifiers or inconsistent entity mappings are flagged for correction or exclusion only when supported by clear business and data-quality evidence.

# 10. Outlier Assessment & Classification

## Objective
Assess unusual numeric observations without automatically removing them.

Extreme values may represent:
- Genuine data-quality errors
- Measurement issues
- Valid but unusual operating conditions
- Operational inefficiencies that should remain available for analysis

The purpose of this stage is therefore to identify and classify unusual observations rather than automatically exclude them.

### 10.1 Numeric Extreme-Value Profile

Review selected percentiles for important manufacturing variables to understand their distributions and identify unusually low or high observations that may require contextual investigation.

In [118]:
outlier_columns = [
    "ItemsProduced",
    "RejectedItems",
    "ActualCycleTimeSec",
    "PowerAvgKW",
    "PowerMaxKW",
    "PowerMinKW",
    "EnergyKWh",
    "AmbientTempC"
]

outlier_profile = fact_production_clean[
    outlier_columns
].quantile(
    [
        0.001,
        0.01,
        0.05,
        0.50,
        0.95,
        0.99,
        0.999
    ]
).T

outlier_profile

,0.00,0.01,0.05,0.50,0.95,0.99,1.00
ItemsProduced,0.00,0.00,0.00,5.00,7.00,7.00,8.00
RejectedItems,0.00,0.00,0.00,0.00,0.00,1.00,1.00
ActualCycleTimeSec,35.52,37.45,39.48,49.39,63.34,67.59,74.08
PowerAvgKW,1.87,2.17,3.83,30.46,38.96,40.83,42.01
PowerMaxKW,2.05,2.38,4.22,33.49,43.02,45.18,46.92
PowerMinKW,1.64,1.90,3.38,26.77,34.49,36.27,37.77
EnergyKWh,0.16,0.18,0.32,2.54,3.25,3.40,3.50
AmbientTempC,15.09,16.02,17.25,24.99,32.75,33.98,34.90


### 10.2 Contextual Energy Extreme Assessment

Assess unusually high or low energy observations relative to comparable records from the same machine and operating status.

Contextual extremes are retained and flagged for later investigation rather than treated as invalid data.

In [119]:
energy_context_stats = (
    fact_production_clean
    .groupby(
        ["MachineID", "StatusID"],
        dropna=False
    )["EnergyKWh"]
    .quantile([0.01, 0.99])
    .unstack()
    .rename(
        columns={
            0.01: "Energy_P01",
            0.99: "Energy_P99"
        }
    )
    .reset_index()
)

energy_context_stats.head()

,MachineID,StatusID,Energy_P01,Energy_P99
0,MCH-001,0,0.27,0.32
1,MCH-001,1,2.18,2.51
2,MCH-001,2,2.59,2.97
3,MCH-001,3,0.52,0.61
4,MCH-002,0,0.29,0.34


In [120]:
fact_production_clean = fact_production_clean.merge(
    energy_context_stats,
    on=["MachineID", "StatusID"],
    how="left"
)

In [121]:
fact_production_clean["OUT_Energy_Contextual"] = (
    (fact_production_clean["EnergyKWh"]
     < fact_production_clean["Energy_P01"])
    |
    (fact_production_clean["EnergyKWh"]
     > fact_production_clean["Energy_P99"])
)

In [122]:
fact_production_clean.drop(
    columns=[
        "Energy_P01",
        "Energy_P99"
    ],
    inplace=True
)

### 10.3 Contextual Cycle-Time Extreme Assessment

Assess unusually high or low cycle-time observations within comparable machine-product operating contexts.

These observations are retained because unusually high cycle time may represent real manufacturing inefficiency rather than poor data quality.

In [123]:
productive_cycle_data = fact_production_clean[
    fact_production_clean["StatusID"].isin(
        productive_status_ids
    )
].copy()

In [124]:
cycle_context_stats = (
    productive_cycle_data
    .groupby(
        ["MachineID", "ProductID"],
        dropna=False
    )["ActualCycleTimeSec"]
    .quantile([0.01, 0.99])
    .unstack()
    .rename(
        columns={
            0.01: "Cycle_P01",
            0.99: "Cycle_P99"
        }
    )
    .reset_index()
)

cycle_context_stats.head()

,MachineID,ProductID,Cycle_P01,Cycle_P99
0,MCH-001,PRD-001,38.84,49.69
1,MCH-001,PRD-002,44.39,56.68
2,MCH-001,PRD-003,50.87,64.94
3,MCH-001,PRD-010,37.00,47.27
4,MCH-002,PRD-001,40.05,51.22


In [125]:
cycle_outlier_check = productive_cycle_data[
    ["ReadingID", "MachineID", "ProductID", "ActualCycleTimeSec"]
].merge(
    cycle_context_stats,
    on=["MachineID", "ProductID"],
    how="left"
)

cycle_outlier_check["OUT_CycleTime_Contextual"] = (
    (cycle_outlier_check["ActualCycleTimeSec"]
     < cycle_outlier_check["Cycle_P01"])
    |
    (cycle_outlier_check["ActualCycleTimeSec"]
     > cycle_outlier_check["Cycle_P99"])
)

In [126]:
cycle_outlier_map = (
    cycle_outlier_check
    .set_index("ReadingID")[
        "OUT_CycleTime_Contextual"
    ]
)

fact_production_clean["OUT_CycleTime_Contextual"] = (
    fact_production_clean["ReadingID"]
    .map(cycle_outlier_map)
    .fillna(False)
    .astype(bool)
)

### Outlier Assessment Decision

Statistical extremes are not automatically removed from the dataset.

Energy and cycle-time observations that are unusual relative to comparable operating contexts are retained and flagged as potential operational anomalies.

Only values that violate hard physical, logical, or structural rules are considered potential data-quality errors.

# 11. Data Quality Flagging

## Objective
Create reusable row-level data-quality indicators based on the validation rules applied in previous stages.

Flags preserve transparency by identifying problematic or unusual observations without automatically deleting them.

In [127]:
fact_production_clean["DQ_InvalidQuantity"] = (
    (fact_production_clean["ItemsProduced"] < 0)
    |
    (fact_production_clean["RejectedItems"] < 0)
    |
    (
        fact_production_clean["RejectedItems"]
        > fact_production_clean["ItemsProduced"]
    )
)

In [128]:
fact_production_clean["DQ_InvalidPower"] = (
    (fact_production_clean["PowerAvgKW"] < 0)
    |
    (fact_production_clean["PowerMaxKW"] < 0)
    |
    (fact_production_clean["PowerMinKW"] < 0)
    |
    ~(
        (
            fact_production_clean["PowerMinKW"]
            <= fact_production_clean["PowerAvgKW"]
        )
        &
        (
            fact_production_clean["PowerAvgKW"]
            <= fact_production_clean["PowerMaxKW"]
        )
    )
)

In [129]:
fact_production_clean["DQ_InvalidEnergy"] = (
    fact_production_clean["EnergyKWh"] < 0
)

In [130]:
fact_production_clean["DQ_InvalidCycleTime"] = (
    (
        fact_production_clean[
            "StandardCycleTimeSec"
        ].notna()
    )
    &
    (
        fact_production_clean[
            "StandardCycleTimeSec"
        ] <= 0
    )
) | (
    fact_production_clean["StatusID"]
    .isin(productive_status_ids)
    &
    (
        fact_production_clean[
            "ActualCycleTimeSec"
        ].isna()
        |
        (
            fact_production_clean[
                "ActualCycleTimeSec"
            ] <= 0
        )
    )
)

In [131]:
fact_production_clean[
    "DQ_DuplicateMachineTimestamp"
] = (
    fact_production_clean
    .duplicated(
        subset=["MachineID", "Timestamp"],
        keep=False
    )
)

### 11.2 Referential Integrity Flag

Create a single row-level indicator identifying production observations whose foreign-key values do not correspond to valid dimension records.

In [132]:
fact_production_clean["DQ_InvalidForeignKey"] = (
    ~fact_production_clean["PlantID"]
    .isin(dim_plant_clean["PlantID"])
    |
    ~fact_production_clean["ShiftID"]
    .isin(dim_shift_clean["ShiftID"])
    |
    ~fact_production_clean["StatusID"]
    .isin(dim_status_clean["StatusID"])
    |
    ~fact_production_clean["TariffID"]
    .isin(dim_tariff_clean["TariffID"])
)

### 11.3 Shift Assignment Quality Flag

Create a row-level indicator identifying observations where the recorded ShiftID does not match the shift expected from the observation timestamp.

This flag helps detect potential inconsistencies in shift assignment that could affect shift-based performance comparisons.

In [133]:
fact_production_clean["DQ_ShiftMismatch"] = (
    fact_production_clean["ShiftID"]
    != fact_production_clean["ExpectedShiftID"]
)

### 11.4 Productive Zero-Output Warning

Flag observations where the machine is recorded in a productive status but no completed units are reported.

These records are treated as operational warnings rather than hard data errors because a five-minute observation interval may end before a production cycle is completed.

In [134]:
fact_production_clean["DQ_ZeroOutputWarning"] = (
    fact_production_clean["StatusID"]
    .isin(productive_status_ids)
    &
    (fact_production_clean["ItemsProduced"] == 0)
)

### 11.5 Consolidated Hard-Error Flag

Combine validation rules representing hard structural or logical failures into a single indicator.

Operational warnings and statistical outliers are intentionally excluded from this flag.

In [135]:
hard_error_columns = [
    "DQ_InvalidQuantity",
    "DQ_InvalidPower",
    "DQ_InvalidEnergy",
    "DQ_InvalidCycleTime",
    "DQ_DuplicateMachineTimestamp",
    "DQ_InvalidForeignKey",
    "DQ_ShiftMismatch"
]

fact_production_clean["DQ_HasError"] = (
    fact_production_clean[hard_error_columns]
    .fillna(False)
    .eq(True)
    .any(axis=1)
)

fact_production_clean["DQ_HasError"].value_counts(dropna=False)

DQ_HasError
False    2947392
Name: count, dtype: Int64

### 11.6 Operational Anomaly Flag

Combine contextual energy and cycle-time outlier indicators into a single operational-anomaly flag.

These observations are retained because they may represent genuine manufacturing inefficiencies rather than data-quality errors.

In [136]:
fact_production_clean["DQ_OperationalAnomaly"] = (
    fact_production_clean["OUT_Energy_Contextual"]
    |
    fact_production_clean["OUT_CycleTime_Contextual"]
)

### 11.7 Assign Overall Data Quality Status

Assign a final row-level data-quality classification based on the previously created validation and anomaly flags.

The classification distinguishes between:
- Hard data-quality errors
- Operational anomalies
- Operational warnings
- Valid observations

In [137]:
conditions = [
    fact_production_clean["DQ_HasError"].fillna(False),
    fact_production_clean["DQ_OperationalAnomaly"].fillna(False),
    fact_production_clean["DQ_ZeroOutputWarning"].fillna(False)
]

choices = [
    "Data Error",
    "Operational Anomaly",
    "Data Warning"
]

fact_production_clean["DQ_Status"] = np.select(
    conditions,
    choices,
    default="Valid"
)

fact_production_clean["DQ_Status"] = (
    fact_production_clean["DQ_Status"]
    .astype("category")
)

### 11.8 Data Quality Classification Summary

Summarize the number and percentage of production-energy observations assigned to each overall data-quality category.

In [138]:
dq_summary = (
    fact_production_clean["DQ_Status"]
    .value_counts(dropna=False)
    .rename_axis("DQ_Status")
    .reset_index(name="Rows")
)

dq_summary["Percent"] = (
    dq_summary["Rows"]
    / len(fact_production_clean)
    * 100
).round(2)

dq_summary

,DQ_Status,Rows,Percent
0,Valid,2836920,96.25
1,Operational Anomaly,110472,3.75


### 11.9 Create Downtime Data Quality Flags

Create row-level data-quality indicators for downtime events based on time ordering, recorded duration, and reference-table relationships.

In [139]:
fact_downtime_clean["DQ_InvalidTimeOrder"] = (
    fact_downtime_clean["EndTimestamp"]
    < fact_downtime_clean["StartTimestamp"]
)

fact_downtime_clean["DQ_InvalidDuration"] = (
    (fact_downtime_clean["DurationMinutes"] <= 0)
    |
    downtime_duration_mismatch
)

fact_downtime_clean["DQ_InvalidForeignKey"] = (
    ~fact_downtime_clean["PlantID"].isin(
        dim_plant_clean["PlantID"]
    )
    |
    ~fact_downtime_clean["ShiftID"].isin(
        dim_shift_clean["ShiftID"]
    )
    |
    ~fact_downtime_clean["ReasonID"].isin(
        dim_downtime_reason_clean["ReasonID"]
    )
)

### 11.10 Assign Downtime Data Quality Status

Combine downtime validation flags into an overall quality indicator for each downtime event.

In [140]:
fact_downtime_clean["DQ_HasError"] = (
    fact_downtime_clean[
        [
            "DQ_InvalidTimeOrder",
            "DQ_InvalidDuration",
            "DQ_InvalidForeignKey"
        ]
    ]
    .fillna(False)
    .any(axis=1)
)

fact_downtime_clean["DQ_Status"] = np.where(
    fact_downtime_clean["DQ_HasError"],
    "Data Error",
    "Valid"
)

fact_downtime_clean["DQ_Status"] = (
    fact_downtime_clean["DQ_Status"]
    .astype("category")
)

### 11.11 Review Data Quality Flags

Review the frequency of each Boolean data-quality and operational-outlier flag.

Only Boolean indicator columns are included in the flag summary.  
The categorical `DQ_Status` field is excluded because it represents an overall classification rather than an individual True/False validation flag.

In [141]:
from pandas.api.types import is_bool_dtype

production_flag_columns = [
    col
    for col in fact_production_clean.columns
    if (
        (col.startswith("DQ_") or col.startswith("OUT_"))
        and is_bool_dtype(fact_production_clean[col])
    )
]

production_flag_summary = pd.DataFrame({
    "Flag": production_flag_columns,
    "Affected_Rows": [
        fact_production_clean[col]
        .fillna(False)
        .sum()
        for col in production_flag_columns
    ]
})

production_flag_summary

,Flag,Affected_Rows
0,OUT_Energy_Contextual,58293
1,OUT_CycleTime_Contextual,53668
2,DQ_InvalidQuantity,0
3,DQ_InvalidPower,0
4,DQ_InvalidEnergy,0
5,DQ_InvalidCycleTime,0
6,DQ_DuplicateMachineTimestamp,0
7,DQ_InvalidForeignKey,0
8,DQ_ShiftMismatch,0
9,DQ_ZeroOutputWarning,0


In [142]:
fact_production_clean[production_flag_columns].dtypes

OUT_Energy_Contextual              bool
OUT_CycleTime_Contextual           bool
DQ_InvalidQuantity              boolean
DQ_InvalidPower                    bool
DQ_InvalidEnergy                   bool
DQ_InvalidCycleTime             boolean
DQ_DuplicateMachineTimestamp       bool
DQ_InvalidForeignKey            boolean
DQ_ShiftMismatch                   bool
DQ_ZeroOutputWarning            boolean
DQ_HasError                     boolean
DQ_OperationalAnomaly              bool
dtype: object

### 11.12 Review Downtime Data Quality Flags

Summarize the number of downtime events affected by each Boolean data-quality flag.

Only Boolean indicators are included in this summary.

In [143]:
downtime_flag_columns = [
    col
    for col in fact_downtime_clean.columns
    if (
        col.startswith("DQ_")
        and is_bool_dtype(fact_downtime_clean[col])
    )
]

downtime_flag_summary = pd.DataFrame({
    "Flag": downtime_flag_columns,
    "Affected_Rows": [
        fact_downtime_clean[col]
        .fillna(False)
        .sum()
        for col in downtime_flag_columns
    ]
})

downtime_flag_summary

,Flag,Affected_Rows
0,DQ_InvalidTimeOrder,0
1,DQ_InvalidDuration,0
2,DQ_InvalidForeignKey,0
3,DQ_HasError,0


### 11.13 Validate Tariff Assignment

Verify that each recorded TariffID matches the tariff expected from the observation year and time-of-day.

This validation ensures that energy-cost calculations use the correct tariff assignment.

In [144]:
fact_production_clean["ExpectedTariffID"].isna().sum()

KeyError: 'ExpectedTariffID'

In [ ]:
fact_production_clean["DQ_TariffMismatch"] = (
    fact_production_clean["ExpectedTariffID"].notna()
    &
    (
        fact_production_clean["TariffID"]
        != fact_production_clean["ExpectedTariffID"]
    )
)

fact_production_clean["DQ_TariffMismatch"].value_counts(dropna=False)

### 11.14 Update Final Data Quality Classification

Update the consolidated data-quality classification after adding tariff-assignment validation.

In [ ]:
hard_error_columns = [
    "DQ_InvalidQuantity",
    "DQ_InvalidPower",
    "DQ_InvalidEnergy",
    "DQ_InvalidCycleTime",
    "DQ_DuplicateMachineTimestamp",
    "DQ_InvalidForeignKey",
    "DQ_ShiftMismatch",
    "DQ_TariffMismatch"
]

fact_production_clean["DQ_HasError"] = (
    fact_production_clean[hard_error_columns]
    .fillna(False)
    .any(axis=1)
)

In [ ]:
conditions = [
    fact_production_clean["DQ_HasError"].fillna(False),
    fact_production_clean["DQ_OperationalAnomaly"].fillna(False),
    fact_production_clean["DQ_ZeroOutputWarning"].fillna(False)
]

choices = [
    "Data Error",
    "Operational Anomaly",
    "Data Warning"
]

fact_production_clean["DQ_Status"] = np.select(
    conditions,
    choices,
    default="Valid"
)

fact_production_clean["DQ_Status"] = (
    fact_production_clean["DQ_Status"]
    .astype("category")
)

### 11.15 Review Final Data Quality Classification

Review the final distribution of production records across the data-quality categories after completing all validation rules.

In [ ]:
dq_summary = (
    fact_production_clean["DQ_Status"]
    .value_counts(dropna=False)
    .rename_axis("DQ_Status")
    .reset_index(name="Rows")
)

dq_summary["Percent"] = (
    dq_summary["Rows"]
    / len(fact_production_clean)
    * 100
).round(2)

dq_summary

# 12. Data Normalization

## Objective
Reduce repeated descriptive attributes in the production fact table by creating dedicated dimensions for machines, production lines, and products.

This prepares the data for SQL Server and Power BI relational modeling.

### 12.1 Create Machine Dimension

Create one unique record per machine containing its descriptive attributes and related line and plant identifiers.

In [146]:
dim_machine_clean = (
    fact_production_clean[
        [
            "MachineID",
            "MachineName",
            "MachineType",
            "LineID",
            "PlantID"
        ]
    ]
    .drop_duplicates()
    .sort_values("MachineID")
    .reset_index(drop=True)
)

dim_machine_clean

,MachineID,MachineName,MachineType,LineID,PlantID
0,MCH-001,CNC Mill 01,CNC Milling,LN-01,PLT-01
1,MCH-002,CNC Mill 02,CNC Milling,LN-01,PLT-01
2,MCH-003,CNC Mill 03,CNC Milling,LN-01,PLT-01
3,MCH-004,CNC Lathe 01,CNC Turning,LN-02,PLT-01
4,MCH-005,CNC Lathe 02,CNC Turning,LN-02,PLT-01
5,MCH-006,CNC Lathe 03,CNC Turning,LN-02,PLT-01
6,MCH-007,Assembly Cell 01,Assembly Cell,LN-03,PLT-01
7,MCH-008,Assembly Cell 02,Assembly Cell,LN-03,PLT-01
8,MCH-009,CNC Mill 04,CNC Milling,LN-04,PLT-02
9,MCH-010,CNC Mill 05,CNC Milling,LN-04,PLT-02


### 12.2 Create Line Dimension

Create one unique record per production line containing its descriptive attributes and related plant identifier.

In [147]:
dim_line_clean = (
    fact_production_clean[
        [
            "LineID",
            "LineName",
            "PlantID"
        ]
    ]
    .drop_duplicates()
    .sort_values("LineID")
    .reset_index(drop=True)
)

dim_line_clean

,LineID,LineName,PlantID
0,LN-01,Precision Machining A,PLT-01
1,LN-02,Precision Machining B,PLT-01
2,LN-03,Assembly & Inspection,PLT-01
3,LN-04,Precision Machining C,PLT-02
4,LN-05,Finishing & Final Assembly,PLT-02


### 12.3 Create Product Dimension

Create one unique record per product containing its descriptive attributes and standard cycle time.

Rows with missing ProductID are excluded because they represent valid non-production states.

In [148]:
dim_product_clean = (
    fact_production_clean.loc[
        fact_production_clean["ProductID"].notna(),
        [
            "ProductID",
            "ProductName",
            "ProductFamily",
            "StandardCycleTimeSec"
        ]
    ]
    .drop_duplicates()
    .sort_values("ProductID")
    .reset_index(drop=True)
)

dim_product_clean

,ProductID,ProductName,ProductFamily,StandardCycleTimeSec
0,PRD-001,Valve Body A,Valve Components,42
1,PRD-002,Valve Body B,Valve Components,48
2,PRD-003,Valve Housing C,Valve Components,55
3,PRD-004,Drive Shaft A,Shaft Components,38
4,PRD-005,Drive Shaft B,Shaft Components,44
5,PRD-006,Rotor Sleeve,Shaft Components,51
6,PRD-007,Mounting Bracket A,Structural Components,46
7,PRD-008,Mounting Bracket B,Structural Components,53
8,PRD-009,Support Frame,Structural Components,62
9,PRD-010,Pump Cover,Housing Components,40


# 13. Feature Engineering

## Objective
Create the core analytical features required to evaluate production quality, cycle-time performance, energy efficiency, energy cost, and temporal patterns.

These features will support the downstream manufacturing-efficiency analysis and contextual benchmarking.

### 13.1 Production Quality Features

Calculate good production output and reject rate to measure production quality.

In [ ]:
fact_production_clean["GoodUnits"] = (
    fact_production_clean["ItemsProduced"]
    - fact_production_clean["RejectedItems"]
)

fact_production_clean["RejectRate"] = (
    fact_production_clean["RejectedItems"]
    / fact_production_clean["ItemsProduced"].replace(0, pd.NA)
)

### 13.2 Cycle-Time Performance Features

Compare actual cycle time with the product standard to identify process-speed deviations under production conditions.

In [ ]:
fact_production_clean["CycleTimeGapSec"] = (
    fact_production_clean["ActualCycleTimeSec"]
    - fact_production_clean["StandardCycleTimeSec"]
)

fact_production_clean["CycleTimeDeviationPct"] = (
    fact_production_clean["CycleTimeGapSec"]
    / fact_production_clean["StandardCycleTimeSec"].replace(0, pd.NA)
    * 100
)

### 13.3 Energy Efficiency Feature

Calculate energy consumption per good unit as a baseline energy-efficiency indicator.

This metric will later be complemented by contextual benchmarking and expected-energy analysis.

In [ ]:
fact_production_clean["EnergyPerGoodUnit"] = (
    fact_production_clean["EnergyKWh"]
    / fact_production_clean["GoodUnits"].replace(0, pd.NA)
)

### 13.4 Energy Cost Feature

Map the electricity tariff rate to each production-energy observation and calculate its associated energy cost.

In [ ]:
tariff_rate_map = (
    dim_tariff_clean
    .set_index("TariffID")["RatePerKWh"]
)

fact_production_clean["RatePerKWh"] = (
    fact_production_clean["TariffID"]
    .map(tariff_rate_map)
)

fact_production_clean["EnergyCost"] = (
    fact_production_clean["EnergyKWh"]
    * fact_production_clean["RatePerKWh"]
)

In [ ]:
print(
    "Missing tariff rates:",
    fact_production_clean["RatePerKWh"].isna().sum()
)

print(
    "Missing energy costs:",
    fact_production_clean["EnergyCost"].isna().sum()
)

### 13.5 Time Features

Derive calendar attributes from Timestamp to support trend, recurrence, and time-series analysis.

In [ ]:
fact_production_clean["Date"] = (
    fact_production_clean["Timestamp"].dt.normalize()
)

fact_production_clean["Year"] = (
    fact_production_clean["Timestamp"].dt.year
)

fact_production_clean["Month"] = (
    fact_production_clean["Timestamp"].dt.month
)

fact_production_clean["Week"] = (
    fact_production_clean["Timestamp"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)

fact_production_clean["Hour"] = (
    fact_production_clean["Timestamp"].dt.hour
)

### 13.6 Validate Engineered Features

Review the newly created analytical features and verify that their missing values are consistent with the relevant operating conditions.

In [ ]:
engineered_features = [
    "GoodUnits",
    "RejectRate",
    "CycleTimeGapSec",
    "CycleTimeDeviationPct",
    "EnergyPerGoodUnit",
    "RatePerKWh",
    "EnergyCost",
    "Date",
    "Year",
    "Month",
    "Week",
    "Hour"
]

feature_validation = pd.DataFrame({
    "Feature": engineered_features,
    "Missing_Rows": [
        fact_production_clean[col].isna().sum()
        for col in engineered_features
    ]
})

feature_validation

### 13.7 Create Final Normalized Production Fact

Create the final production fact table by removing descriptive attributes stored in dimensions and temporary validation fields while preserving identifiers, measures, engineered features, and data-quality flags.

In [149]:
fact_production_final = (
    fact_production_clean
    .drop(
        columns=[
            "MachineName",
            "MachineType",
            "LineName",
            "ProductName",
            "ProductFamily",
            "StandardCycleTimeSec",
            "ExpectedShiftID",
            "ExpectedTariffID",
            "MinuteOfDay"
        ],
        errors="ignore"
    )
    .copy()
)

print("Final production shape:", fact_production_final.shape)

Final production shape: (2947392, 30)


In [ ]:
required_objects = [
    "fact_production_final",
    "fact_downtime_clean",
    "dim_machine_clean",
    "dim_line_clean",
    "dim_product_clean"
]

for obj in required_objects:
    print(obj, "OK" if obj in globals() else "MISSING")

# 14. Final Validation

## Objective
Confirm that the prepared fact and dimension tables are structurally valid and ready for export to the SQL Server preparation layer.

### 14.1 Validate Record Preservation and Keys

Confirm that data preparation did not unintentionally remove fact records and that primary identifiers remain unique.

In [ ]:
final_validation = pd.DataFrame({
    "Check": [
        "Production rows removed",
        "Downtime rows removed",
        "Duplicate ReadingID",
        "Duplicate DowntimeEventID",
        "Duplicate MachineID",
        "Duplicate LineID",
        "Duplicate ProductID"
    ],

    "Result": [
        len(fact_production) - len(fact_production_final),
        len(fact_downtime) - len(fact_downtime_clean),
        fact_production_final["ReadingID"].duplicated().sum(),
        fact_downtime_clean["DowntimeEventID"].duplicated().sum(),
        dim_machine_clean["MachineID"].duplicated().sum(),
        dim_line_clean["LineID"].duplicated().sum(),
        dim_product_clean["ProductID"].duplicated().sum()
    ]
})

final_validation

### 14.2 Final Data Quality Summary

Review the final distribution of production records across the overall data-quality classifications.

In [ ]:
final_quality_summary = (
    fact_production_final["DQ_Status"]
    .value_counts(dropna=False)
    .rename_axis("DQ_Status")
    .reset_index(name="Rows")
)

final_quality_summary["Percent"] = (
    final_quality_summary["Rows"]
    / len(fact_production_final)
    * 100
).round(2)

final_quality_summary

In [ ]:
dq_cols = [
    "DQ_InvalidQuantity",
    "DQ_InvalidPower",
    "DQ_InvalidEnergy",
    "DQ_InvalidCycleTime",
    "DQ_DuplicateMachineTimestamp",
    "DQ_InvalidForeignKey",
    "DQ_ShiftMismatch",
    "DQ_ZeroOutputWarning",
    "DQ_HasError",
    "DQ_OperationalAnomaly"
]

for col in dq_cols:
    print(col, fact_production_final[col].value_counts(dropna=False))

### 14.3 Final Safety Check

Confirm that record counts, primary keys, and normalized dimensions remain structurally valid before exporting the prepared datasets.ة

In [ ]:
assert len(fact_production_final) == len(fact_production), (
    "Production row count changed unexpectedly."
)

assert len(fact_downtime_clean) == len(fact_downtime), (
    "Downtime row count changed unexpectedly."
)

assert fact_production_final["ReadingID"].is_unique, (
    "ReadingID contains duplicates."
)

assert fact_downtime_clean["DowntimeEventID"].is_unique, (
    "DowntimeEventID contains duplicates."
)

assert dim_machine_clean["MachineID"].is_unique
assert dim_line_clean["LineID"].is_unique
assert dim_product_clean["ProductID"].is_unique

print("Final validation passed successfully.")

# 15. Export Prepared Data

## Objective

Export the validated and normalized datasets for the next stage of the project in SQL Server, while preserving the final data-quality summaries for documentation.

### 15.1 Prepare Export Directories

Ensure that the processed-data and output directories exist before exporting the prepared datasets and validation reports.

In [ ]:
PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Processed path:", PROCESSED_PATH)
print("Output path:", OUTPUT_PATH)

### 15.2 Export Dimension Tables

Export the cleaned and normalized dimension tables that will form the descriptive layer of the SQL Server data model.

In [ ]:
dimension_exports = {
    "Dim_Plant.csv": dim_plant_clean,
    "Dim_Shift.csv": dim_shift_clean,
    "Dim_Status.csv": dim_status_clean,
    "Dim_EnergyTariff.csv": dim_tariff_clean,
    "Dim_DowntimeReason.csv": dim_downtime_reason_clean,
    "Dim_Machine.csv": dim_machine_clean,
    "Dim_Line.csv": dim_line_clean,
    "Dim_Product.csv": dim_product_clean
}

for file_name, dataframe in dimension_exports.items():
    dataframe.to_csv(
        PROCESSED_PATH / file_name,
        index=False
    )

print(
    f"{len(dimension_exports)} dimension tables exported successfully."
)

### 15.3 Export Downtime Fact

Export the validated downtime-event fact table for downstream SQL Server modeling and analysis.

In [ ]:
fact_downtime_clean.to_csv(
    PROCESSED_PATH / "Fact_DowntimeEvents.csv",
    index=False
)

print(
    "Downtime fact exported:",
    len(fact_downtime_clean),
    "rows"
)

### 15.4 Export Production-Energy Fact

Export the final production-energy fact table in Parquet format to efficiently preserve the large prepared dataset before loading it into SQL Server.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
production_export_file = (
    PROCESSED_PATH / "Fact_ProductionEnergy.csv"
)

fact_production_final.to_csv(
    production_export_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print(
    "Production fact exported:",
    len(fact_production_final),
    "rows"
)

print(
    "File size:",
    round(
        production_export_file.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

### 15.5 Export Data Quality Reports

Export the final data-quality and validation summaries to preserve evidence of the data preparation process for project documentation.

In [ ]:
final_quality_summary.to_csv(
    OUTPUT_PATH / "Production_DQ_Status_Summary.csv",
    index=False
)

production_flag_summary.to_csv(
    OUTPUT_PATH / "Production_DQ_Flag_Summary.csv",
    index=False
)

downtime_flag_summary.to_csv(
    OUTPUT_PATH / "Downtime_DQ_Flag_Summary.csv",
    index=False
)

final_validation.to_csv(
    OUTPUT_PATH / "Final_Validation_Summary.csv",
    index=False
)

print("Data quality reports exported successfully.")

### 15.6 Verify Exported Files

Verify that all processed datasets and data-quality reports were successfully exported before closing the data preparation stage.

In [ ]:
print("PROCESSED DATA")
print("-" * 50)

for file in sorted(PROCESSED_PATH.iterdir()):
    print(
        f"{file.name:<35}",
        f"{file.stat().st_size / (1024 ** 2):,.2f} MB"
    )

print("\nOUTPUT REPORTS")
print("-" * 50)

for file in sorted(OUTPUT_PATH.iterdir()):
    print(
        f"{file.name:<35}",
        f"{file.stat().st_size / 1024:,.2f} KB"
    )

# 16. Data Preparation Complete

The manufacturing datasets have been successfully profiled, cleaned, validated, normalized, and enriched with analytical features.

The final analytical structure contains:

- 8 dimension tables
- 2 fact tables
- Data-quality indicators and validation reports
- Production, quality, cycle-time, energy-efficiency, cost, and temporal features

Operational anomalies were intentionally preserved for investigation during the analytical stage rather than being removed as data errors.

## Next Stage

**SQL Server — Data Modeling, Relationships, Analytical Views, and KPI Preparation**

In [ ]:
for var in [
    "dim_machine_clean",
    "dim_line_clean",
    "dim_product_clean"
]:
    print(var, var in globals())

In [ ]:
sql_ready_tables = {
    "Dim_Plant": dim_plant_clean,
    "Dim_Shift": dim_shift_clean,
    "Dim_Status": dim_status_clean,
    "Dim_EnergyTariff": dim_tariff_clean,
    "Dim_DowntimeReason": dim_downtime_reason_clean,
    "Dim_Machine": dim_machine_clean,
    "Dim_Line": dim_line_clean,
    "Dim_Product": dim_product_clean,
    "Fact_DowntimeEvents": fact_downtime_clean,
    "Fact_ProductionEnergy": fact_production_final
}

for table_name, df in sql_ready_tables.items():

    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)

    print("Rows:", f"{len(df):,}")
    print("Columns:", len(df))

    print("\nCOLUMN SCHEMA")

    schema = pd.DataFrame({
        "Column": df.columns,
        "Pandas_Type": df.dtypes.astype(str).values,
        "Null_Count": df.isna().sum().values
    })

    print(schema.to_string(index=False))

In [ ]:
print("fact_production_final" in globals())

## 17. SQL Server Schema Preparation

This section inspects the final processed dimension tables before creating
their physical structure in SQL Server.

The objective is to identify the exact column names, data types, and nullability
required for the SQL relational model.

In [151]:
dimensions_for_sql = {
    "Dim_Plant": dim_plant_clean,
    "Dim_Line": dim_line_clean,
    "Dim_Machine": dim_machine_clean,
    "Dim_Product": dim_product_clean,
    "Dim_Shift": dim_shift_clean,
    "Dim_Status": dim_status_clean,
    "Dim_EnergyTariff": dim_tariff_clean,
    "Dim_DowntimeReason": dim_downtime_reason_clean
}

for table_name, df in dimensions_for_sql.items():

    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)

    schema = pd.DataFrame({
        "Column": df.columns,
        "PandasType": df.dtypes.astype(str).values,
        "NullCount": df.isna().sum().values,
        "UniqueValues": df.nunique(dropna=True).values
    })

    print(schema.to_string(index=False))


Dim_Plant
      Column PandasType  NullCount  UniqueValues
     PlantID        str          0             2
   PlantName        str          0             2
        City        str          0             2
     Country        str          0             1
  GridRegion        str          0             2
IndustryType        str          0             1

Dim_Line
  Column PandasType  NullCount  UniqueValues
  LineID        str          0             5
LineName        str          0             5
 PlantID        str          0             2

Dim_Machine
     Column PandasType  NullCount  UniqueValues
  MachineID        str          0            14
MachineName        str          0            14
MachineType        str          0             4
     LineID        str          0             5
    PlantID        str          0             2

Dim_Product
              Column PandasType  NullCount  UniqueValues
           ProductID        str          0            12
         ProductName        

## 17.1 SQL Server Fact Table Schema Preparation

This section inspects the final processed fact tables before creating
their physical structure in SQL Server.

The objective is to identify the exact columns, data types, nullability,
key fields, and analytical attributes required for the SQL fact tables.

In [152]:
facts_for_sql = {
    "Fact_DowntimeEvents": fact_downtime_clean,
    "Fact_ProductionEnergy": fact_production_final
}

for table_name, df in facts_for_sql.items():

    print("\n" + "=" * 90)
    print(table_name)
    print("=" * 90)

    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")

    schema = pd.DataFrame({
        "Column": df.columns,
        "PandasType": df.dtypes.astype(str).values,
        "NullCount": df.isna().sum().values,
        "UniqueValues": df.nunique(dropna=True).values
    })

    print("\nCOLUMN SCHEMA")
    print(schema.to_string(index=False))


Fact_DowntimeEvents
Rows: 13,046
Columns: 16

COLUMN SCHEMA
                 Column     PandasType  NullCount  UniqueValues
        DowntimeEventID            str          0         13046
                PlantID            str          0             2
              MachineID            str          0            14
                 LineID            str          0             5
                ShiftID            str          0             3
         StartTimestamp datetime64[us]          0         12424
           EndTimestamp datetime64[us]          0         12417
        DurationMinutes          Int64          0             6
               ReasonID            str          0             6
              ProductID            str          0            12
EnergyDuringDowntimeKWh        float64          0          3405
    DQ_InvalidTimeOrder           bool          0             1
     DQ_InvalidDuration        boolean          0             1
   DQ_InvalidForeignKey           bool     

## 18. SQL Server Data Loading

The validated and processed datasets are now ready to be loaded into the
SQL Server relational model.

The SQL tables have already been created with the required data types,
primary keys, and foreign-key relationships.

Python will therefore be used only to load the processed records into the
existing database structure.

In [153]:
import sqlalchemy
import pyodbc

print("SQLAlchemy:", sqlalchemy.__version__)
print("pyodbc:", pyodbc.version)

SQLAlchemy: 2.0.52
pyodbc: 5.3.0


In [154]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'PostgreSQL ODBC Driver(ANSI)', 'PostgreSQL ODBC Driver(UNICODE)']


### 18.1  SQL Server Connection

This step establishes a trusted connection between Python and the
ManufacturingEfficiencyDB database in SQL Server.

Windows Authentication is used, and the connection is tested before
loading any processed data.

In [5]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

server = r"A-FF-L6-9"
database = "ManufacturingEfficiencyDB"

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}"
)

In [165]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                @@SERVERNAME AS ServerName,
                DB_NAME() AS CurrentDatabase
        """)
    )

    row = result.fetchone()

    print("Server:", row.ServerName)
    print("Database:", row.CurrentDatabase)

Server: DESKTOP-JD0PB6T
Database: ManufacturingEfficiencyDB


### 18.2 Load the First Dimension Table

This step performs a controlled test load of the `Dim_Plant` table into SQL Server.

The objective is to verify that:

- The Python DataFrame structure matches the SQL table structure.
- Column names and data types are compatible.
- The Python-to-SQL insertion process works correctly.
- Existing primary key and foreign key constraints remain preserved.

The load uses `append` mode because the SQL table has already been created manually with the required schema and constraints.

In [167]:
dim_plant_clean.to_sql(
    name="Dim_Plant",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False
)

print("Dim_Plant loaded successfully.")

Dim_Plant loaded successfully.


### 18.3 Validate the Dim_Plant Load

This step validates the first Python-to-SQL data transfer.

The number of rows stored in SQL Server is compared with the processed Python DataFrame, and the loaded records are reviewed to confirm that the data was transferred correctly.

In [168]:
print("Python rows:", len(dim_plant_clean))

Python rows: 2


### 18.4 Load the Remaining Dimension Tables

After successfully validating the first test load, the remaining dimension tables can now be transferred to SQL Server.

The dimensions are loaded before the fact tables because they are referenced by foreign keys in the fact tables.

Each DataFrame is appended to its corresponding pre-created SQL table so that the existing schema, primary keys, foreign keys, and data types remain unchanged.

In [169]:
dimension_tables = {
    "Dim_Line": dim_line_clean,
    "Dim_Machine": dim_machine_clean,
    "Dim_Product": dim_product_clean,
    "Dim_Shift": dim_shift_clean,
    "Dim_Status": dim_status_clean,
    "Dim_EnergyTariff": dim_tariff_clean,
    "Dim_DowntimeReason": dim_downtime_reason_clean
}

for table_name, df in dimension_tables.items():
    df.to_sql(
        name=table_name,
        con=engine,
        schema="dbo",
        if_exists="append",
        index=False
    )

    print(f"{table_name} loaded successfully.")

Dim_Line loaded successfully.
Dim_Machine loaded successfully.
Dim_Product loaded successfully.
Dim_Shift loaded successfully.
Dim_Status loaded successfully.
Dim_EnergyTariff loaded successfully.
Dim_DowntimeReason loaded successfully.


### 18.5 Validate All Dimension Loads

This step verifies that all dimension tables were loaded completely and correctly into SQL Server.

The row counts in Python are compared with the corresponding row counts in SQL Server to confirm that no records were lost or duplicated during the transfer.

In [171]:
dimension_validation = []

all_dimensions = {
    "Dim_Plant": dim_plant_clean,
    "Dim_Line": dim_line_clean,
    "Dim_Machine": dim_machine_clean,
    "Dim_Product": dim_product_clean,
    "Dim_Shift": dim_shift_clean,
    "Dim_Status": dim_status_clean,
    "Dim_EnergyTariff": dim_tariff_clean,
    "Dim_DowntimeReason": dim_downtime_reason_clean
}

for table_name, df in all_dimensions.items():

    sql_count = pd.read_sql(
        f"SELECT COUNT(*) AS [RowCount] FROM dbo.{table_name}",
        engine
    )["RowCount"].iloc[0]

    python_count = len(df)

    dimension_validation.append({
        "Table": table_name,
        "PythonRows": python_count,
        "SQLRows": sql_count,
        "Match": python_count == sql_count
    })

dimension_validation_df = pd.DataFrame(dimension_validation)

dimension_validation_df

,Table,PythonRows,SQLRows,Match
0,Dim_Plant,2,2,True
1,Dim_Line,5,5,True
2,Dim_Machine,14,14,True
3,Dim_Product,12,12,True
4,Dim_Shift,3,3,True
5,Dim_Status,4,4,True
6,Dim_EnergyTariff,8,8,True
7,Dim_DowntimeReason,6,6,True


### 18.6 Load the Downtime Fact Table

This step loads the cleaned downtime event data from Python into the existing `dbo.Fact_DowntimeEvents` table in SQL Server.

The dimension tables have already been loaded, so the foreign key relationships can now be enforced during the insert process.

The existing SQL table structure is preserved by using append mode.

In [174]:
fact_downtime_clean.to_sql(
    name="Fact_DowntimeEvents",
    con=engine,
    schema="dbo",
    if_exists="append",
    index=False,
    chunksize=1000
)

print("dbo.Fact_DowntimeEvents loaded successfully.")

dbo.Fact_DowntimeEvents loaded successfully.


### 18.7 Validate the Downtime Fact Load

This step validates the downtime fact transfer by comparing the number of records in the cleaned Python DataFrame with the number of records stored in SQL Server.

Matching row counts confirm that the downtime records were transferred completely.

In [176]:
python_downtime_rows = len(fact_downtime_clean)

sql_downtime_rows = pd.read_sql(
    """
    SELECT COUNT(*) AS [RowCount]
    FROM dbo.Fact_DowntimeEvents;
    """,
    engine
)["RowCount"].iloc[0]

print("Python rows:", python_downtime_rows)
print("SQL rows:", sql_downtime_rows)
print("Match:", python_downtime_rows == sql_downtime_rows)

Python rows: 13046
SQL rows: 13046
Match: True


### 18.8 Preview the Loaded Downtime Data

This step retrieves a sample of the downtime records directly from SQL Server.

The preview confirms that identifiers, timestamps, downtime durations, energy values, and data quality fields were stored correctly.

In [177]:
downtime_sql_preview = pd.read_sql(
    """
    SELECT TOP 10 *
    FROM dbo.Fact_DowntimeEvents
    ORDER BY StartTimestamp;
    """,
    engine
)

downtime_sql_preview

,DowntimeEventID,PlantID,MachineID,LineID,ShiftID,StartTimestamp,EndTimestamp,DurationMinutes,ReasonID,ProductID,EnergyDuringDowntimeKWh,DQ_InvalidTimeOrder,DQ_InvalidDuration,DQ_InvalidForeignKey,DQ_HasError,DQ_Status
0,DT-011227,PLT-02,MCH-013,LN-05,S3,2024-01-01 00:35:00,2024-01-01 01:00:00,25,R06,PRD-011,1.97,False,False,False,False,Valid
1,DT-006557,PLT-01,MCH-008,LN-03,S3,2024-01-01 00:45:00,2024-01-01 00:50:00,5,R02,PRD-009,0.29,False,False,False,False,Valid
2,DT-010325,PLT-02,MCH-012,LN-05,S3,2024-01-01 02:15:00,2024-01-01 02:30:00,15,R03,PRD-007,1.08,False,False,False,False,Valid
3,DT-003750,PLT-01,MCH-005,LN-02,S3,2024-01-01 02:25:00,2024-01-01 02:45:00,20,R02,PRD-011,2.11,False,False,False,False,Valid
4,DT-001862,PLT-01,MCH-003,LN-01,S3,2024-01-01 02:30:00,2024-01-01 02:45:00,15,R02,PRD-003,1.75,False,False,False,False,Valid
5,DT-000001,PLT-01,MCH-001,LN-01,S3,2024-01-01 02:50:00,2024-01-01 03:05:00,15,R01,PRD-002,1.57,False,False,False,False,Valid
6,DT-004701,PLT-01,MCH-006,LN-02,S3,2024-01-01 02:50:00,2024-01-01 03:05:00,15,R05,PRD-005,1.62,False,False,False,False,Valid
7,DT-000002,PLT-01,MCH-001,LN-01,S1,2024-01-01 06:35:00,2024-01-01 06:50:00,15,R06,PRD-001,1.79,False,False,False,False,Valid
8,DT-011228,PLT-02,MCH-013,LN-05,S1,2024-01-01 13:05:00,2024-01-01 13:35:00,30,R03,PRD-005,2.50,False,False,False,False,Valid
9,DT-003751,PLT-01,MCH-005,LN-02,S2,2024-01-01 15:40:00,2024-01-01 16:05:00,25,R06,PRD-005,2.65,False,False,False,False,Valid


### 18.9 Prepare the Production Fact Table for SQL Loading

This step prepares the cleaned production-energy DataFrame before transferring it to SQL Server.

Because `fact_production_clean` contains approximately 2.95 million rows, the loading process requires additional preparation to reduce the risk of type conversion issues, invalid NULL handling, or interrupted inserts.

The preparation focuses on validating the final column structure, converting nullable values into SQL-compatible representations, and confirming that the DataFrame matches the pre-created `dbo.Fact_ProductionEnergy` table.

In [178]:
print("Rows:", len(fact_production_clean))
print("Columns:", len(fact_production_clean.columns))

fact_production_clean.dtypes

Rows: 2947392
Columns: 37


ReadingID                                int64
Timestamp                       datetime64[us]
PlantID                                    str
ShiftID                                    str
TariffID                                   str
StatusID                                 Int64
MachineID                                  str
MachineName                                str
MachineType                                str
LineID                                     str
LineName                                   str
ProductID                                  str
ProductName                                str
ProductFamily                              str
StandardCycleTimeSec                     Int64
ItemsProduced                            Int64
RejectedItems                            Int64
ActualCycleTimeSec                     float64
PowerAvgKW                             float64
PowerMaxKW                             float64
PowerMinKW                             float64
EnergyKWh    

### 18.10 Prepare Production Data for SQL

The production dataset is aligned with the existing SQL table structure by retaining only the columns defined in `dbo.Fact_ProductionEnergy`.

This prevents duplicated dimension attributes and temporary validation fields from being loaded into the fact table.

In [182]:
sql_production_columns = pd.read_sql(
    """
    SELECT COLUMN_NAME
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'dbo'
      AND TABLE_NAME = 'Fact_ProductionEnergy'
    ORDER BY ORDINAL_POSITION;
    """,
    engine
)["COLUMN_NAME"].tolist()

fact_production_sql = fact_production_clean[
    sql_production_columns
].copy()

print("Rows:", len(fact_production_sql))
print("Columns:", len(fact_production_sql.columns))
print(
    "Column match:",
    fact_production_sql.columns.tolist() == sql_production_columns
)

Rows: 2947392
Columns: 30
Column match: True


In [185]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT
                @@SERVERNAME AS ServerName,
                DB_NAME() AS CurrentDatabase
        """)
    )

    row = result.fetchone()

    print("Server:", row.ServerName)
    print("Database:", row.CurrentDatabase)

Server: DESKTOP-JD0PB6T
Database: ManufacturingEfficiencyDB


### 18.11 Create a Fast SQL Server Connection

A SQL Server engine with `fast_executemany` is created to improve the performance of the large production fact load.

In [186]:
fast_engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}",
    fast_executemany=True
)

print("Fast SQL Server engine created successfully.")

Fast SQL Server engine created successfully.


### 18.12 Load the Production Fact in Controlled Batches

The production fact is loaded in separate batches so that progress can be tracked and each completed batch is committed independently.

This approach reduces the risk of losing the entire load if the process is interrupted.

In [187]:
total_rows = len(fact_production_sql)
batch_size = 50000

with fast_engine.connect() as connection:
    current_rows = connection.execute(
        text("""
            SELECT COUNT(*) AS [RowCount]
            FROM dbo.Fact_ProductionEnergy
        """)
    ).scalar()

if current_rows != 0:
    raise ValueError(
        f"Fact_ProductionEnergy already contains {current_rows:,} rows."
    )

for start in range(0, total_rows, batch_size):

    end = min(start + batch_size, total_rows)

    batch = fact_production_sql.iloc[start:end]

    batch.to_sql(
        name="Fact_ProductionEnergy",
        con=fast_engine,
        schema="dbo",
        if_exists="append",
        index=False,
        chunksize=5000
    )

    print(
        f"Loaded {end:,} / {total_rows:,} rows "
        f"({end / total_rows:.1%})"
    )

print("Fact_ProductionEnergy loaded successfully.")

Loaded 50,000 / 2,947,392 rows (1.7%)
Loaded 100,000 / 2,947,392 rows (3.4%)
Loaded 150,000 / 2,947,392 rows (5.1%)
Loaded 200,000 / 2,947,392 rows (6.8%)
Loaded 250,000 / 2,947,392 rows (8.5%)
Loaded 300,000 / 2,947,392 rows (10.2%)
Loaded 350,000 / 2,947,392 rows (11.9%)
Loaded 400,000 / 2,947,392 rows (13.6%)
Loaded 450,000 / 2,947,392 rows (15.3%)
Loaded 500,000 / 2,947,392 rows (17.0%)
Loaded 550,000 / 2,947,392 rows (18.7%)
Loaded 600,000 / 2,947,392 rows (20.4%)
Loaded 650,000 / 2,947,392 rows (22.1%)
Loaded 700,000 / 2,947,392 rows (23.7%)
Loaded 750,000 / 2,947,392 rows (25.4%)
Loaded 800,000 / 2,947,392 rows (27.1%)
Loaded 850,000 / 2,947,392 rows (28.8%)
Loaded 900,000 / 2,947,392 rows (30.5%)
Loaded 950,000 / 2,947,392 rows (32.2%)
Loaded 1,000,000 / 2,947,392 rows (33.9%)
Loaded 1,050,000 / 2,947,392 rows (35.6%)
Loaded 1,100,000 / 2,947,392 rows (37.3%)
Loaded 1,150,000 / 2,947,392 rows (39.0%)
Loaded 1,200,000 / 2,947,392 rows (40.7%)
Loaded 1,250,000 / 2,947,392 rows (4

### 18.14 Validate the Production Fact Load

This step validates the production fact load by comparing the total number of records in Python with the records stored in SQL Server.

In [188]:
python_rows = len(fact_production_sql)

sql_rows = pd.read_sql(
    """
    SELECT COUNT(*) AS [RowCount]
    FROM dbo.Fact_ProductionEnergy;
    """,
    fast_engine
)["RowCount"].iloc[0]

print("Python rows:", python_rows)
print("SQL rows:", sql_rows)
print("Match:", python_rows == sql_rows)

Python rows: 2947392
SQL rows: 2947392
Match: True


In [8]:
query = """
SELECT
    ProductionDate,
    PlantID,
    LineID,
    MachineID,
    MachineType,
    ProductID,
    ProductFamily,
    ShiftID,
    GoodItems,
    RejectedItems,
    RejectRate,
    AvgCycleTimeSec,
    AvgCycleTimeDeviationSec,
    TotalEnergyKWh,
    ProductiveEnergyKWh,
    NonProductiveEnergyKWh,
    TotalEnergyCost,
    OperationalAnomalyCount,
    DowntimeEventCount,
    TotalDowntimeMinutes,
    UnplannedDowntimeMinutes
FROM analytics.vw_OperationalPerformance
WHERE
    GoodItems > 0
    AND ProductID IS NOT NULL;
"""

ml_df = pd.read_sql(query, engine)

print("Rows:", len(ml_df))
print("Columns:", ml_df.shape[1])

ml_df.head()

Rows: 40936
Columns: 21


,ProductionDate,PlantID,LineID,MachineID,MachineType,ProductID,ProductFamily,ShiftID,GoodItems,RejectedItems,...,AvgCycleTimeSec,AvgCycleTimeDeviationSec,TotalEnergyKWh,ProductiveEnergyKWh,NonProductiveEnergyKWh,TotalEnergyCost,OperationalAnomalyCount,DowntimeEventCount,TotalDowntimeMinutes,UnplannedDowntimeMinutes
0,2024-05-13,PLT-02,LN-04,MCH-011,CNC Milling,PRD-001,Valve Components,S3,385,2,...,47.055396,5.055396,205.0924,201.8731,3.2193,225.601640,0,1,25,25
1,2025-04-05,PLT-02,LN-05,MCH-013,Finishing Cell,PRD-008,Structural Components,S3,322,1,...,55.167352,2.167352,130.6146,130.6146,0.0000,163.268250,0,0,0,0
2,2025-01-08,PLT-01,LN-01,MCH-002,CNC Milling,PRD-002,Valve Components,S2,473,5,...,53.251833,5.251833,271.8205,271.8205,0.0000,507.447225,0,0,0,0
3,2025-09-10,PLT-02,LN-05,MCH-012,Finishing Cell,PRD-004,Shaft Components,S3,442,3,...,40.984558,2.984558,116.0328,116.0328,0.0000,145.041000,4,0,0,0
4,2025-11-09,PLT-02,LN-05,MCH-014,Finishing Cell,PRD-006,Shaft Components,S3,343,4,...,51.637058,0.637058,121.2865,121.2865,0.0000,151.608125,1,0,0,0


In [9]:
print("Dataset Shape:")
print(ml_df.shape)

print("\nData Types:")
print(ml_df.dtypes)

print("\nMissing Values:")
print(
    ml_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nDuplicate Rows:")
print(ml_df.duplicated().sum())

print("\nTarget Distribution - TotalEnergyKWh:")
print(
    ml_df["TotalEnergyKWh"]
    .describe()
)

Dataset Shape:
(40936, 21)

Data Types:
ProductionDate               object
PlantID                         str
LineID                          str
MachineID                       str
MachineType                     str
ProductID                       str
ProductFamily                   str
ShiftID                         str
GoodItems                     int64
RejectedItems                 int64
RejectRate                  float64
AvgCycleTimeSec             float64
AvgCycleTimeDeviationSec    float64
TotalEnergyKWh              float64
ProductiveEnergyKWh         float64
NonProductiveEnergyKWh      float64
TotalEnergyCost             float64
OperationalAnomalyCount       int64
DowntimeEventCount            int64
TotalDowntimeMinutes          int64
UnplannedDowntimeMinutes      int64
dtype: object

Missing Values:
ProductionDate              0
PlantID                     0
LineID                      0
MachineID                   0
MachineType                 0
ProductID              

In [10]:
target = "TotalEnergyKWh"

categorical_features = [
    "PlantID",
    "LineID",
    "MachineID",
    "MachineType",
    "ProductID",
    "ProductFamily",
    "ShiftID"
]

numeric_features = [
    "GoodItems",
    "RejectedItems",
    "RejectRate",
    "AvgCycleTimeSec",
    "AvgCycleTimeDeviationSec"
]

excluded_features = [
    "ProductionDate",
    "ProductiveEnergyKWh",
    "NonProductiveEnergyKWh",
    "TotalEnergyCost",
    "OperationalAnomalyCount",
    "DowntimeEventCount",
    "TotalDowntimeMinutes",
    "UnplannedDowntimeMinutes"
]

model_features = categorical_features + numeric_features

X = ml_df[model_features].copy()
y = ml_df[target].copy()

print("Target:")
print(target)

print("\nModel Features:")
print(model_features)

print("\nExcluded Features:")
print(excluded_features)

print("\nX Shape:", X.shape)
print("y Shape:", y.shape)

Target:
TotalEnergyKWh

Model Features:
['PlantID', 'LineID', 'MachineID', 'MachineType', 'ProductID', 'ProductFamily', 'ShiftID', 'GoodItems', 'RejectedItems', 'RejectRate', 'AvgCycleTimeSec', 'AvgCycleTimeDeviationSec']

Excluded Features:
['ProductionDate', 'ProductiveEnergyKWh', 'NonProductiveEnergyKWh', 'TotalEnergyCost', 'OperationalAnomalyCount', 'DowntimeEventCount', 'TotalDowntimeMinutes', 'UnplannedDowntimeMinutes']

X Shape: (40936, 12)
y Shape: (40936,)


In [11]:
ml_df["ProductionDate"] = pd.to_datetime(ml_df["ProductionDate"])

split_date = pd.Timestamp("2025-07-01")

train_mask = ml_df["ProductionDate"] < split_date
test_mask = ml_df["ProductionDate"] >= split_date

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Split Date:", split_date.date())

print("\nTraining Set:")
print("Rows:", len(X_train))
print(
    "Date Range:",
    ml_df.loc[train_mask, "ProductionDate"].min().date(),
    "to",
    ml_df.loc[train_mask, "ProductionDate"].max().date()
)

print("\nTest Set:")
print("Rows:", len(X_test))
print(
    "Date Range:",
    ml_df.loc[test_mask, "ProductionDate"].min().date(),
    "to",
    ml_df.loc[test_mask, "ProductionDate"].max().date()
)

print("\nTraining Share:")
print(f"{len(X_train) / len(X) * 100:.2f}%")

print("\nTest Share:")
print(f"{len(X_test) / len(X) * 100:.2f}%")

Split Date: 2025-07-01

Training Set:
Rows: 30632
Date Range: 2024-01-01 to 2025-06-30

Test Set:
Rows: 10304
Date Range: 2025-07-01 to 2025-12-31

Training Share:
74.83%

Test Share:
25.17%


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Original Training Shape:", X_train.shape)
print("Processed Training Shape:", X_train_processed.shape)

print("\nOriginal Test Shape:", X_test.shape)
print("Processed Test Shape:", X_test_processed.shape)

Original Training Shape: (30632, 12)
Processed Training Shape: (30632, 49)

Original Test Shape: (10304, 12)
Processed Test Shape: (10304, 49)


## 24.6 Establish a Baseline Model

This step establishes a simple reference model before evaluating machine-learning algorithms.

The baseline predicts the median training energy consumption for all future observations. More advanced models must outperform this benchmark to demonstrate meaningful predictive value. 

In [13]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline_model = DummyRegressor(strategy="median")

baseline_model.fit(
    X_train_processed,
    y_train
)

baseline_predictions = baseline_model.predict(
    X_test_processed
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_predictions
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_predictions
)

print("Baseline Model Performance")
print("--------------------------")
print(f"MAE:  {baseline_mae:.2f} kWh")
print(f"RMSE: {baseline_rmse:.2f} kWh")
print(f"R²:   {baseline_r2:.4f}")

Baseline Model Performance
--------------------------
MAE:  71.31 kWh
RMSE: 84.10 kWh
R²:   -0.0005


## 24.7 Compare Regression Models

This step compares tree-based regression models for estimating expected energy consumption under different manufacturing operating contexts.

The models are evaluated on the future test period using MAE, RMSE, and R², and their performance is compared with the previously established baseline. 

In [14]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
        min_samples_leaf=5
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        min_samples_leaf=5,
        n_jobs=-1
    )
}

results = []

for model_name, model in models.items():

    model.fit(
        X_train_processed,
        y_train
    )

    predictions = model.predict(
        X_test_processed
    )

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    results.append({
        "Model": model_name,
        "MAE_kWh": mae,
        "RMSE_kWh": rmse,
        "R2": r2
    })

model_results = pd.DataFrame(results)

model_results

,Model,MAE_kWh,RMSE_kWh,R2
0,Decision Tree,1.241838,1.912459,0.999483
1,Random Forest,1.127946,1.734953,0.999574


## 24.8 Perform a Model Sanity Check

This step verifies that the strong Random Forest performance reflects genuine predictive relationships rather than an unintended feature leakage pattern.

The analysis examines feature importance to confirm which operating-context variables are driving expected-energy predictions. 

In [15]:
final_model = models["Random Forest"]

feature_names = preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": final_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(15)

,Feature,Importance
44,numeric__GoodItems,0.653077
22,categorical__MachineType_CNC Milling,0.170658
43,categorical__ShiftID_S3,0.084242
23,categorical__MachineType_CNC Turning,0.039923
3,categorical__LineID_LN-02,0.032867
47,numeric__AvgCycleTimeSec,0.003396
0,categorical__PlantID_PLT-01,0.001869
1,categorical__PlantID_PLT-02,0.001863
10,categorical__MachineID_MCH-004,0.001813
5,categorical__LineID_LN-04,0.001662


## 24.9 Generate Expected Energy Predictions

This step applies the selected Random Forest model to the future test period to estimate expected energy consumption.

The predicted energy is compared with actual consumption to quantify the ML-based efficiency gap for each operating context. 

In [16]:
final_predictions = final_model.predict(X_test_processed)

ml_results = ml_df.loc[test_mask].copy()

ml_results["ML_ExpectedEnergyKWh"] = final_predictions

ml_results["ML_EnergyGapKWh"] = (
    ml_results["TotalEnergyKWh"]
    - ml_results["ML_ExpectedEnergyKWh"]
)

ml_results["ML_EnergyGapPercent"] = (
    ml_results["ML_EnergyGapKWh"]
    / ml_results["ML_ExpectedEnergyKWh"]
)

print(
    ml_results[
        [
            "ProductionDate",
            "MachineID",
            "ProductID",
            "ShiftID",
            "GoodItems",
            "TotalEnergyKWh",
            "ML_ExpectedEnergyKWh",
            "ML_EnergyGapKWh",
            "ML_EnergyGapPercent"
        ]
    ].head(10)
)

   ProductionDate MachineID ProductID ShiftID  GoodItems  TotalEnergyKWh  \
3      2025-09-10   MCH-012   PRD-004      S3        442        116.0328   
4      2025-11-09   MCH-014   PRD-006      S3        343        121.2865   
6      2025-07-03   MCH-006   PRD-008      S2        440        251.8435   
10     2025-08-29   MCH-003   PRD-010      S2        574        246.8183   
33     2025-07-11   MCH-014   PRD-007      S3        392        118.1061   
41     2025-12-20   MCH-014   PRD-012      S2        478        161.9627   
42     2025-12-12   MCH-011   PRD-012      S1        471        299.4982   
43     2025-10-14   MCH-009   PRD-010      S2        584        273.8464   
47     2025-08-17   MCH-002   PRD-001      S3        411        196.8682   
48     2025-11-14   MCH-005   PRD-005      S1        510        224.2068   

    ML_ExpectedEnergyKWh  ML_EnergyGapKWh  ML_EnergyGapPercent  
3             116.024453         0.008347             0.000072  
4             120.766769         

## 24.10 Define a Meaningful ML Energy Gap Threshold

This step distinguishes meaningful excess-energy deviations from normal model prediction error.

A threshold is derived from the distribution of absolute prediction errors in the future test period. Contexts exceeding the selected error threshold are flagged for further efficiency investigation. 

In [17]:
absolute_errors = np.abs(
    y_test.values - final_predictions
)

error_p95 = np.percentile(
    absolute_errors,
    95
)

print(
    f"95th Percentile Absolute Error: "
    f"{error_p95:.2f} kWh"
)

ml_results["ML_MeaningfulExcessFlag"] = (
    ml_results["ML_EnergyGapKWh"] > error_p95
).astype(int)

print("\nMeaningful ML Excess Contexts:")
print(
    ml_results["ML_MeaningfulExcessFlag"]
    .value_counts()
    .sort_index()
)

print("\nMeaningful Excess Energy:")
print(
    ml_results.loc[
        ml_results["ML_MeaningfulExcessFlag"] == 1,
        "ML_EnergyGapKWh"
    ].sum()
)

95th Percentile Absolute Error: 3.39 kWh

Meaningful ML Excess Contexts:
ML_MeaningfulExcessFlag
0    10096
1      208
Name: count, dtype: int64

Meaningful Excess Energy:
1043.3229033194805


## 24.11 Calibrate the ML Gap Threshold

This step calibrates the meaningful prediction-error threshold using historical data only.

A recent portion of the training period is used as a calibration window, keeping the final test period fully unseen when the threshold is defined. 

In [18]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor

calibration_date = pd.Timestamp("2025-04-01")

development_mask = (
    (ml_df["ProductionDate"] < calibration_date)
)

calibration_mask = (
    (ml_df["ProductionDate"] >= calibration_date)
    & (ml_df["ProductionDate"] < split_date)
)

X_dev = X.loc[development_mask].copy()
y_dev = y.loc[development_mask].copy()

X_cal = X.loc[calibration_mask].copy()
y_cal = y.loc[calibration_mask].copy()

calibration_preprocessor = clone(preprocessor)

X_dev_processed = calibration_preprocessor.fit_transform(X_dev)
X_cal_processed = calibration_preprocessor.transform(X_cal)

calibration_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    min_samples_leaf=5,
    n_jobs=-1
)

calibration_model.fit(
    X_dev_processed,
    y_dev
)

calibration_predictions = calibration_model.predict(
    X_cal_processed
)

calibration_absolute_errors = np.abs(
    y_cal.values - calibration_predictions
)

final_error_threshold = np.percentile(
    calibration_absolute_errors,
    95
)

print("Development Rows:", len(X_dev))
print("Calibration Rows:", len(X_cal))

print(
    f"Final P95 Error Threshold: "
    f"{final_error_threshold:.2f} kWh"
)

Development Rows: 25536
Calibration Rows: 5096
Final P95 Error Threshold: 3.57 kWh


In [19]:
ml_results["ML_MeaningfulExcessFlag"] = (
    ml_results["ML_EnergyGapKWh"]
    > final_error_threshold
).astype(int)

flagged_contexts = (
    ml_results["ML_MeaningfulExcessFlag"].sum()
)

flagged_excess_energy = ml_results.loc[
    ml_results["ML_MeaningfulExcessFlag"] == 1,
    "ML_EnergyGapKWh"
].sum()

print("\nFinal Test Results:")
print("Flagged Contexts:", flagged_contexts)
print(
    f"Flagged Excess Energy: "
    f"{flagged_excess_energy:.2f} kWh"
)


Final Test Results:
Flagged Contexts: 179
Flagged Excess Energy: 942.55 kWh


## 24.12 Prepare the Final ML Decision-Support Output

This step prepares a compact analytical output containing validated future-period expected-energy predictions and meaningful excess-energy flags.

The output preserves the operational context identifiers required for integration with SQL Server and Power BI. 

In [20]:
ml_output = ml_results[
    [
        "ProductionDate",
        "PlantID",
        "LineID",
        "MachineID",
        "ProductID",
        "ShiftID",
        "GoodItems",
        "TotalEnergyKWh",
        "ML_ExpectedEnergyKWh",
        "ML_EnergyGapKWh",
        "ML_EnergyGapPercent",
        "ML_MeaningfulExcessFlag"
    ]
].copy()

ml_output["ML_ErrorThresholdKWh"] = final_error_threshold

print("ML Output Shape:", ml_output.shape)

print("\nFlag Distribution:")
print(
    ml_output["ML_MeaningfulExcessFlag"]
    .value_counts()
    .sort_index()
)

ml_output.head()

ML Output Shape: (10304, 13)

Flag Distribution:
ML_MeaningfulExcessFlag
0    10125
1      179
Name: count, dtype: int64


,ProductionDate,PlantID,LineID,MachineID,ProductID,ShiftID,GoodItems,TotalEnergyKWh,ML_ExpectedEnergyKWh,ML_EnergyGapKWh,ML_EnergyGapPercent,ML_MeaningfulExcessFlag,ML_ErrorThresholdKWh
3,2025-09-10,PLT-02,LN-05,MCH-012,PRD-004,S3,442,116.0328,116.024453,0.008347,0.000072,0,3.567054
4,2025-11-09,PLT-02,LN-05,MCH-014,PRD-006,S3,343,121.2865,120.766769,0.519731,0.004304,0,3.567054
6,2025-07-03,PLT-01,LN-02,MCH-006,PRD-008,S2,440,251.8435,251.103475,0.740025,0.002947,0,3.567054
10,2025-08-29,PLT-01,LN-01,MCH-003,PRD-010,S2,574,246.8183,247.628837,-0.810537,-0.003273,0,3.567054
33,2025-07-11,PLT-02,LN-05,MCH-014,PRD-007,S3,392,118.1061,118.001787,0.104313,0.000884,0,3.567054


## 24.13 Write the ML Results Back to SQL Server

This step stores the validated machine-learning output in the SQL analytical layer.

The resulting table can be integrated with Power BI as a supplementary predictive decision-support layer alongside the historical contextual benchmark. 

In [21]:
ml_output.to_sql(
    name="ML_EnergyPredictions",
    con=engine,
    schema="analytics",
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("ML results successfully written to SQL Server.")

ML results successfully written to SQL Server.


## 24.14 Validate ML Value Against the Historical Contextual Benchmark

This step validates whether the selected Random Forest model provides predictive value beyond the project's simpler historical contextual benchmark.

To ensure a fair comparison, production-volume boundaries and benchmark energy values are derived from the training period only and then applied to the unseen test period. The Random Forest and contextual benchmark are evaluated on the same covered test observations using MAE, RMSE, and R².

This validation does not modify the final model. It verifies whether the machine-learning layer adds measurable value over the interpretable historical benchmark.

In [22]:


import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# 1. Prepare train and test datasets
benchmark_train = ml_df.loc[train_mask].copy()
benchmark_test = ml_df.loc[test_mask].copy()

# Add the already-generated Random Forest predictions
benchmark_test["RF_ExpectedEnergyKWh"] = final_predictions


# 2. Calculate production-volume boundaries from TRAINING DATA ONLY
volume_boundaries = (
    benchmark_train
    .groupby(
        ["ProductID", "MachineType"]
    )["GoodItems"]
    .quantile([0.25, 0.50, 0.75])
    .unstack()
    .reset_index()
)

volume_boundaries.columns = [
    "ProductID",
    "MachineType",
    "P25",
    "P50",
    "P75"
]


# 3. Attach the historical boundaries to train and test
benchmark_train = benchmark_train.merge(
    volume_boundaries,
    on=["ProductID", "MachineType"],
    how="left"
)

benchmark_test = benchmark_test.merge(
    volume_boundaries,
    on=["ProductID", "MachineType"],
    how="left"
)


# 4. Assign Production Volume Bands using TRAINING boundaries
def assign_volume_band(row):

    if row["GoodItems"] <= row["P25"]:
        return "Low"

    elif row["GoodItems"] <= row["P50"]:
        return "Medium"

    elif row["GoodItems"] <= row["P75"]:
        return "High"

    else:
        return "Very High"


benchmark_train["ProductionVolumeBand"] = (
    benchmark_train.apply(
        assign_volume_band,
        axis=1
    )
)

benchmark_test["ProductionVolumeBand"] = (
    benchmark_test.apply(
        assign_volume_band,
        axis=1
    )
)


# 5. Build the historical contextual benchmark from TRAINING DATA ONLY
benchmark_table = (
    benchmark_train
    .groupby(
        [
            "ProductID",
            "MachineType",
            "ShiftID",
            "ProductionVolumeBand"
        ]
    )
    .agg(
        Benchmark_ExpectedEnergyKWh=(
            "TotalEnergyKWh",
            "median"
        ),
        BenchmarkContextCount=(
            "TotalEnergyKWh",
            "size"
        )
    )
    .reset_index()
)


# Keep only sufficiently supported benchmark groups
benchmark_table = benchmark_table.loc[
    benchmark_table["BenchmarkContextCount"] >= 10
].copy()


# 6. Apply the historical benchmark to the unseen TEST period
benchmark_test = benchmark_test.merge(
    benchmark_table,
    on=[
        "ProductID",
        "MachineType",
        "ShiftID",
        "ProductionVolumeBand"
    ],
    how="left"
)


# 7. Evaluate both methods on EXACTLY the same covered test observations
comparison_df = benchmark_test.loc[
    benchmark_test["Benchmark_ExpectedEnergyKWh"].notna()
].copy()

y_actual = comparison_df["TotalEnergyKWh"]

benchmark_predictions = (
    comparison_df["Benchmark_ExpectedEnergyKWh"]
)

rf_predictions = (
    comparison_df["RF_ExpectedEnergyKWh"]
)


# Historical Contextual Benchmark metrics
benchmark_mae = mean_absolute_error(
    y_actual,
    benchmark_predictions
)

benchmark_rmse = np.sqrt(
    mean_squared_error(
        y_actual,
        benchmark_predictions
    )
)

benchmark_r2 = r2_score(
    y_actual,
    benchmark_predictions
)


# Random Forest metrics on the same rows
rf_mae = mean_absolute_error(
    y_actual,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_actual,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_actual,
    rf_predictions
)


# 8. Calculate benchmark coverage
coverage_percent = (
    len(comparison_df)
    / len(benchmark_test)
    * 100
)


# 9. Quantify Random Forest improvement
mae_improvement = (
    (benchmark_mae - rf_mae)
    / benchmark_mae
    * 100
)

rmse_improvement = (
    (benchmark_rmse - rf_rmse)
    / benchmark_rmse
    * 100
)


# 10. Present the final comparison
validation_results = pd.DataFrame(
    {
        "Method": [
            "Historical Contextual Benchmark",
            "Random Forest"
        ],
        "MAE_kWh": [
            benchmark_mae,
            rf_mae
        ],
        "RMSE_kWh": [
            benchmark_rmse,
            rf_rmse
        ],
        "R2": [
            benchmark_r2,
            rf_r2
        ]
    }
)

print("Covered Test Rows:", len(comparison_df))
print(
    f"Benchmark Coverage: "
    f"{coverage_percent:.2f}%"
)

print(
    f"\nRandom Forest MAE Improvement: "
    f"{mae_improvement:.2f}%"
)

print(
    f"Random Forest RMSE Improvement: "
    f"{rmse_improvement:.2f}%"
)

print("\nFinal Validation Results:")

display(
    validation_results.round(
        {
            "MAE_kWh": 4,
            "RMSE_kWh": 4,
            "R2": 6
        }
    )
)

Covered Test Rows: 10298
Benchmark Coverage: 99.94%

Random Forest MAE Improvement: 81.08%
Random Forest RMSE Improvement: 80.53%

Final Validation Results:


,Method,MAE_kWh,RMSE_kWh,R2
0,Historical Contextual Benchmark,5.9481,8.8703,0.988873
1,Random Forest,1.1254,1.7268,0.999578
